# E7-B Colab — Qwen3.5-27B LoRA 학습·평가 (A100 80GB)

**학습 구성 B:** A에서 결함 13문항(선명한 글자와 정답이 어긋남, 전제가 틀림 등. splits/e7_review_highconf.csv)을 뺀 것 — **5,401문항** (train 5,401 / dev 0)

정답은 여러 교육생이 일치해서 고른 답입니다. 제로샷 모델은 사진 속 사실은 잘 맞히지만
일반인 상식·눈에 띄는 쪽·맥락 해석 같은 **사람의 해석 방식**은 모릅니다. 그것을 train 정답으로 학습시킵니다.

| 단계 | 내용 | 시간 (추정) |
| --- | --- | ---: |
| 설치·데이터·모델 다운로드 | | 15~20분 |
| **학습** | 1 epoch, 유효 배치 16, 체크포인트는 Drive에 | **2.5~4시간** (첫 스텝 뒤 실측 ETA 출력) |
| 병합 | LoRA를 bf16 원본에 합쳐 로컬 디스크에 저장 | 약 10분 |
| 평가 | vLLM으로 holdout 1,300, E6 제로샷과 McNemar | 약 15분 (적재 포함) |
| 제출 (선택) | test 6,714 | 약 22분 |

**세션이 끊겨도 됩니다.** 학습은 20스텝(약 320문항)마다 Drive에 체크포인트를 남기고,
다시 실행하면 거기서 이어 갑니다. 여러 세션에 나눠 돌려도 결과는 같습니다.

## 준비물

- A100 80GB 런타임
- Drive의 `MyDrive/SSAFY/AI_Challenge2/ssafy-16-2-ai.zip`
- 평가 비교용: E6 결과 `MyDrive/SSAFY/AI_Challenge2/output/exp_E6_zeroshot_train/` (없으면 비교만 건너뜀)

공통 코드·학습 구성 목록·holdout ID는 노트북 안에 있습니다.

## 1. 설치

새 런타임에서 가장 먼저 실행하고, 끝나면 **런타임 → 세션 다시 시작** 후 2번 셀부터 실행하세요.
`flash-linear-attention`은 Qwen3.5의 linear attention 층을 빠르게 학습시키는 커널입니다(없으면 매우 느림).

In [ ]:
!pip -q install -U vllm
!pip -q install -U peft bitsandbytes flash-linear-attention "huggingface_hub[hf_xet]" pillow pandas
# Colab 기본 패키지 중 두 개가 충돌합니다. 이 작업에는 둘 다 필요 없으므로 지웁니다.
# - torchaudio: vLLM이 바꾼 torch와 CUDA 버전이 어긋나 transformers import가 실패
# - torchao 0.10: peft가 0.16 미만을 발견하면 LoRA를 붙일 때 ImportError
!pip -q uninstall -y torchaudio torchao

In [ ]:
import gc, json, math, os, random, re, shutil, sys, time
from pathlib import Path

import pandas as pd
import torch
import torchvision
import transformers
import peft
import vllm

print("vllm", vllm.__version__, "/ transformers", transformers.__version__, "/ peft", peft.__version__)
print("torch", torch.__version__, f"(CUDA {torch.version.cuda})", "/ torchvision", torchvision.__version__)
try:
    import torchaudio  # noqa: F401
    raise RuntimeError("torchaudio가 남아 있습니다. `!pip uninstall -y torchaudio` 후 세션을 다시 시작하세요")
except ImportError:
    pass
try:
    import torchao  # noqa: F401
    raise RuntimeError("torchao가 남아 있습니다. `!pip uninstall -y torchao` 후 세션을 다시 시작하세요")
except ImportError:
    pass
try:
    import fla  # noqa: F401
    print("flash-linear-attention: 설치됨")
except ImportError:
    print("경고: flash-linear-attention이 없어 학습이 매우 느립니다")
assert torch.cuda.is_available(), "GPU 런타임을 선택하세요"
GPU_GIB = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f"GPU: {torch.cuda.get_device_name(0)} / {GPU_GIB:.1f} GiB")
assert GPU_GIB >= 70, "27B bf16 학습은 A100 80GB가 필요합니다"

## 2. Drive · 데이터 · 공통 코드

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DRIVE_BASE = Path("/content/drive/MyDrive/SSAFY/AI_Challenge2")
ZIP_PATH = DRIVE_BASE / "ssafy-16-2-ai.zip"
ZIP_PASSWORD = "ocrisnotenough"
assert ZIP_PATH.is_file(), f"zip을 찾지 못했습니다: {ZIP_PATH}"


def find_data_root():
    for parent, dirs, files in os.walk("/content"):
        dirs[:] = [d for d in dirs if d not in {"drive", ".cache", "sample_data", "models"}]
        if "test.csv" in files and "train.csv" in files:
            return Path(parent)
    return None


if find_data_root() is None:
    import subprocess
    subprocess.run(["7z", "x", "-y", f"-p{ZIP_PASSWORD}", "-o/content/", str(ZIP_PATH)],
                   check=True, stdout=subprocess.DEVNULL)
DATA_ROOT = find_data_root()
assert DATA_ROOT is not None and (DATA_ROOT / "dev.csv").is_file(), "train/test/dev.csv를 찾지 못했습니다"

SOURCES = {'common_eval.py': '"""실험 공통 코드.\n\n모든 실험(A: OCR+LLM, B: VLM 단독, C: OCR+VLM, D: LLM blind)이\n동일한 검증 분할 / 질문 프롬프트 / 출력 파서 / 분석 지표를 쓰도록 한곳에 모읍니다.\n각 노트북이 코드를 복사하면 조용히 어긋나므로 반드시 import 해서 사용하세요.\n"""\nimport hashlib\nimport json\nimport re\nfrom pathlib import Path\n\nimport pandas as pd\n\nDATA_DIR = Path("data")\nCHOICES = list("abcd")\nSYSTEM = ("You are a visual question answering assistant. "\n          "Answer with exactly one lowercase letter: a, b, c, or d. No explanation.")\n# 실험 A가 만든 기준 분할. 모든 실험은 이 val 672문항을 그대로 씁니다.\nBASELINE_RUN = Path("output/paddleocr_vl16_halftrain/full")\n\n\nANSWER_INSTRUCTION = "이미지를 보고 정답 선지의 영문 소문자 한 글자만 답하세요."\n# CoT에서는 "한 글자만" 지시와 "단계적으로 설명" 지시가 충돌하므로 마지막 줄을 통째로 교체합니다.\nCOT_INSTRUCTION = ("이미지에서 근거가 되는 부분을 찾아 간단히 단계적으로 설명한 뒤, "\n                   "마지막 줄에 반드시 `정답: x` 형식으로 a, b, c, d 중 한 글자만 쓰세요.")\n\n\n# E6: 정답은 사진 속 사실이 아니라 여러 교육생이 일치해서 고른 답입니다(응답이 갈린 문항은 제외됨).\n# E1 오답 중 \'라벨 오류\'로 본 문항도 사람 기준으로는 일반인 상식(그리스 문자 -> 라틴어)이나\n# 눈에 띄는 쪽(두 수건 중 앞의 흰 수건)을 고른 것이었습니다. 그 기준을 지시문으로 알려 봅니다.\nCONSENSUS_INSTRUCTION = ("이 문제의 정답은 여러 사람이 같은 사진을 보고 일치해서 고른 답입니다. "\n                         "사진에 적힌 표기를 그대로 따르고, 전문 지식보다 일반적인 상식과 "\n                         "눈에 잘 띄는 정보를 기준으로 사람들이 가장 많이 고를 선지의 "\n                         "영문 소문자 한 글자만 답하세요.")\n\n\ndef make_prompt(row, cot=False, consensus=False):\n    """질문/선지 렌더링. 본문은 모든 실험에서 동일하고 마지막 지시문만 달라집니다."""\n    assert not (cot and consensus), "CoT와 합의 지시문은 함께 쓰지 않습니다"\n    instruction = COT_INSTRUCTION if cot else CONSENSUS_INSTRUCTION if consensus else ANSWER_INSTRUCTION\n    return (f"질문: {row[\'question\']}\\n"\n            f"a. {row[\'a\']}\\nb. {row[\'b\']}\\nc. {row[\'c\']}\\nd. {row[\'d\']}\\n"\n            + instruction)\n\n\ndef parse_choice(raw):\n    """출력이 정확히 선지 한 글자일 때만 인정하고, 나머지는 INVALID로 남깁니다."""\n    raw = re.sub(r"<think>.*?</think>", "", str(raw), flags=re.S).strip()\n    raw = re.sub(r"^(?:정답|답|answer)\\s*[:：]\\s*", "", raw, flags=re.I).strip()\n    m = re.fullmatch(r"(?:\\(?([abcd])\\)?[.。]?)", raw, flags=re.I)\n    return m.group(1).lower() if m else "INVALID"\n\n\n# CoT 모드에서는 system 프롬프트의 \'설명 금지\'를 풀어야 합니다.\nSYSTEM_COT = ("You are a visual question answering assistant. "\n              "Think briefly, then end your reply with a final line "\n              "in the form `정답: x` where x is one of a, b, c, d.")\n\n_FINAL_RE = re.compile(r"(?:정답|답|answer|final answer)\\s*[:：]\\s*\\(?\\s*([abcd])\\b",\n                       re.IGNORECASE)\n_BARE_RE = re.compile(r"(?<![a-z])\\(?([abcd])\\)?(?![a-z])", re.IGNORECASE)\n\n\ndef parse_choice_cot(raw):\n    """CoT 출력에서 최종 선지를 추출합니다.\n\n    parse_choice()는 출력이 정확히 한 글자일 때만 인정하므로 CoT 문장은 전부 INVALID가 됩니다.\n    여기서는 `정답: x` 형태를 우선 찾고, 없으면 마지막 줄의 단독 a~d를 씁니다.\n    추론 과정에 등장하는 선지 언급을 최종 답으로 오인하지 않도록 뒤에서부터 찾습니다.\n    """\n    text = re.sub(r"<think>.*?</think>", " ", str(raw), flags=re.S)\n    matches = _FINAL_RE.findall(text)\n    if matches:\n        return matches[-1].lower()\n    # `정답:` 표기가 없으면 마지막 비어 있지 않은 줄에서만 단독 글자를 찾습니다.\n    for line in reversed([l.strip() for l in text.splitlines() if l.strip()]):\n        found = _BARE_RE.findall(line)\n        if len(found) == 1:\n            return found[0].lower()\n        if found:\n            break            # 한 줄에 여러 선지가 있으면 최종 답으로 볼 수 없습니다.\n    return "INVALID"\n\n\n# --- E3: 선지를 감추고 먼저 읽게 하는 2단계 프롬프트 -------------------------\n# E1/E2는 질문과 선지 4개를 한꺼번에 줍니다. 그러면 선지가 판독을 유도합니다.\n# 실제로 E1 오답 중 CoT로도 못 고친 22건에서 정답 문자열이 추론문에 등장한 것은 5건뿐이고,\n# 모델은 간판의 \'미술랭투스타\'를 흔한 표기 \'미슐랭 스타\'로 교정해 읽었습니다.\n# 1단계에서 선지를 감추면 그 닻이 사라지는지 보는 것이 E3의 가설입니다.\n\nSYSTEM_READ = ("You are a careful transcription assistant. "\n               "Write down exactly what is printed in the image. "\n               "Never correct spelling, never normalize to a more common wording, never guess.")\n\nREAD_INSTRUCTION = ("위 질문에 답하는 데 필요한 부분의 글자를 이미지에 적힌 그대로 옮겨 적으세요. "\n                    "맞춤법을 고치거나 더 흔한 표현으로 바꾸지 마세요. "\n                    "글자가 보이지 않으면 \'확인 불가\'라고만 쓰세요.")\n\nMATCH_INSTRUCTION = "전사 내용과 이미지를 함께 보고 정답 선지의 영문 소문자 한 글자만 답하세요."\n\n\ndef make_read_prompt(row):\n    """1단계. 선지를 넣지 않습니다. 넣는 순간 실험이 무의미해집니다."""\n    return f"질문: {row[\'question\']}\\n" + READ_INSTRUCTION\n\n\ndef make_match_prompt(row, transcript):\n    """2단계. 1단계 전사를 앞에 붙이고 나머지는 E1과 동일하게 렌더링합니다."""\n    return (f"이미지에서 읽은 글자: {str(transcript).strip()}\\n\\n"\n            f"질문: {row[\'question\']}\\n"\n            f"a. {row[\'a\']}\\nb. {row[\'b\']}\\nc. {row[\'c\']}\\nd. {row[\'d\']}\\n"\n            + MATCH_INSTRUCTION)\n\n\nIMAGE_MAX_EDGE = 960  # 원본 노트북의 공통 이미지 규칙. B와 C가 반드시 같은 값을 써야 비교가 성립합니다.\n\n\ndef open_image(row, max_edge=IMAGE_MAX_EDGE):\n    """원본 노트북 open_image와 동일: RGB 변환 후 긴 변을 max_edge로 제한."""\n    from PIL import Image\n    with Image.open(DATA_DIR / str(row["path"])) as im:\n        im = im.convert("RGB")\n        if max_edge is not None and max(im.size) > max_edge:\n            scale = max_edge / max(im.size)\n            im = im.resize((max(1, round(im.width * scale)), max(1, round(im.height * scale))),\n                           Image.Resampling.LANCZOS)\n        return im\n\n\ndef ids_sha256(ids):\n    return hashlib.sha256(json.dumps(list(ids)).encode()).hexdigest()\n\n\ndef load_val(run_dir=BASELINE_RUN):\n    """기준 실험이 저장한 검증 분할을 그대로 읽고, ID 해시로 동일 분할임을 확인합니다.\n\n    sklearn 버전이 달라져도 재계산하지 않으므로 분할이 어긋날 수 없습니다.\n    """\n    run_dir = Path(run_dir)\n    val_path = run_dir / "train_half_val.csv"\n    if not val_path.is_file():\n        raise FileNotFoundError(\n            f"기준 분할이 없습니다: {val_path}. 먼저 실험 A 노트북의 분할 셀을 실행하세요.")\n    val_df = pd.read_csv(val_path, keep_default_na=False)\n    config_path = run_dir / "run_config.json"\n    if config_path.is_file():\n        expected = json.loads(config_path.read_text(encoding="utf-8")).get("validation_ids_sha256")\n        actual = ids_sha256(val_df.id.tolist())\n        if expected and expected != actual:\n            raise ValueError(f"검증 분할이 기준과 다릅니다: {actual} != {expected}")\n        print(f"검증 분할 해시 일치: {actual[:16]}…")\n    missing = [p for p in val_df.path if not (DATA_DIR / p).is_file()]\n    if missing:\n        raise FileNotFoundError(f"이미지 누락 {len(missing)}개. 예: {missing[:3]}")\n    print(f"검증 문항 {len(val_df)}개 / 정답 분포:\\n{val_df.answer.value_counts().sort_index()}")\n    return val_df\n\n\ndef load_ocr_evidence(run_dir=BASELINE_RUN, mode="ocr_llm"):\n    """실험 A가 남긴 stage1.jsonl(Spotting 결과)을 id -> 텍스트로 읽습니다. 실험 C가 사용."""\n    path = Path(run_dir) / mode / "stage1.jsonl"\n    if not path.is_file():\n        raise FileNotFoundError(f"OCR 결과가 없습니다: {path}. 실험 A를 먼저 완료하세요.")\n    evidence = {}\n    with path.open(encoding="utf-8") as f:\n        for line in f:\n            item = json.loads(line)\n            evidence[item["id"]] = item["stage1_raw"]\n    empty = sum(1 for v in evidence.values() if not v.strip())\n    print(f"OCR 근거 {len(evidence)}건 로드 (빈 결과 {empty}건)")\n    return evidence\n\n\ndef save_analysis(pred_df, name, out_dir, extra=None):\n    """정확도/macro-F1/혼동행렬/분류리포트를 저장하고 지표 dict를 반환합니다."""\n    from sklearn.metrics import (accuracy_score, classification_report,\n                                 confusion_matrix, f1_score)\n    import matplotlib.pyplot as plt\n\n    dest = Path(out_dir)\n    dest.mkdir(parents=True, exist_ok=True)\n    pred_df.to_csv(dest / "validation_predictions.csv", index=False, encoding="utf-8-sig")\n    pred_df.loc[~pred_df["correct"]].to_csv(dest / "validation_errors.csv",\n                                            index=False, encoding="utf-8-sig")\n    # INVALID를 열로 유지하여 오답이 사라지거나 임의로 a가 되지 않도록 합니다.\n    labels = CHOICES + ["INVALID"]\n    cm = confusion_matrix(pred_df["answer"], pred_df["pred"], labels=labels)\n    cm_df = pd.DataFrame(cm, index=[f"true_{k}" for k in labels],\n                         columns=[f"pred_{k}" for k in labels])\n    cm_df.to_csv(dest / "confusion_matrix.csv", encoding="utf-8-sig")\n    class_df = pd.DataFrame(classification_report(\n        pred_df["answer"], pred_df["pred"], labels=CHOICES,\n        output_dict=True, zero_division=0)).T\n    class_df.to_csv(dest / "classification_report.csv", encoding="utf-8-sig")\n    score = {\n        "experiment": name,\n        "validation_rows": len(pred_df),\n        "correct": int(pred_df.correct.sum()),\n        "accuracy": float(accuracy_score(pred_df.answer, pred_df.pred)),\n        "macro_f1": float(f1_score(pred_df.answer, pred_df.pred, labels=CHOICES,\n                                   average="macro", zero_division=0)),\n        "invalid": int((pred_df.pred == "INVALID").sum()),\n        "invalid_rate": float((pred_df.pred == "INVALID").mean()),\n        "mean_inference_seconds": float(pred_df.seconds.mean()),\n        "total_inference_seconds": float(pred_df.seconds.sum()),\n        "validation_ids_sha256": ids_sha256(pred_df.id.tolist()),\n    }\n    score.update(extra or {})\n    (dest / "metrics.json").write_text(json.dumps(score, ensure_ascii=False, indent=2),\n                                       encoding="utf-8")\n    print(f"[{name}] Accuracy={score[\'accuracy\']:.4f} / macro-F1={score[\'macro_f1\']:.4f} "\n          f"/ INVALID={score[\'invalid\']}")\n    print(class_df.loc[CHOICES, ["precision", "recall", "f1-score", "support"]].round(3))\n    fig, ax = plt.subplots(figsize=(7, 5))\n    ax.imshow(cm[:4, :], cmap="Blues")\n    ax.set_xticks(range(5), labels)\n    ax.set_yticks(range(4), CHOICES)\n    ax.set_xlabel("Predicted")\n    ax.set_ylabel("True")\n    ax.set_title(f"{name} | Accuracy={score[\'accuracy\']:.3f}")\n    for i in range(4):\n        for j in range(5):\n            ax.text(j, i, str(cm[i, j]), ha="center", va="center")\n    fig.tight_layout()\n    fig.savefig(dest / "confusion_matrix.png", dpi=180)\n    plt.show()\n    plt.close(fig)\n    return score\n\n\ndef _self_test():\n    assert parse_choice("c") == "c"\n    assert parse_choice("a 또는 b") == "INVALID"\n    assert parse_choice("") == "INVALID"\n    assert parse_choice("정답: D") == "d"\n    assert parse_choice("간판을 보면 ... 정답: c") == "INVALID"   # 엄격 파서는 문장을 거부\n    # CoT 파서\n    assert parse_choice_cot("a 와 b 를 비교하면...\\n정답: c") == "c"\n    assert parse_choice_cot("생각: b 인 것 같지만 아니다.\\n정답: (D)") == "d"\n    assert parse_choice_cot("설명이 길다\\n\\nb") == "b"\n    assert parse_choice_cot("a 또는 b 중 하나") == "INVALID"\n    assert parse_choice_cot("") == "INVALID"\n    assert parse_choice_cot("정답: a 이지만 정답: b 로 정정") == "b"   # 마지막 선언을 채택\n    row = {"question": "Q", "a": "A", "b": "B", "c": "C", "d": "D"}\n    assert make_prompt(row).endswith(ANSWER_INSTRUCTION)\n    assert make_prompt(row, cot=True).endswith(COT_INSTRUCTION)\n    # 질문/선지 본문은 두 모드가 완전히 동일해야 비교가 성립합니다.\n    assert make_prompt(row).rsplit(chr(10), 1)[0] == make_prompt(row, cot=True).rsplit(chr(10), 1)[0]\n    assert make_prompt(row, consensus=True).endswith(CONSENSUS_INSTRUCTION)\n    assert make_prompt(row).rsplit(chr(10), 1)[0] == make_prompt(row, consensus=True).rsplit(chr(10), 1)[0]\n    # E3 1단계는 선지가 새면 안 됩니다.\n    read = make_read_prompt(row)\n    assert read.endswith(READ_INSTRUCTION)\n    for key in CHOICES:\n        assert f"{key}. {row[key]}" not in read, "1단계 프롬프트에 선지가 들어갔습니다"\n    # E3 2단계는 전사문만 앞에 붙고 질문/선지 본문은 E1과 동일해야 합니다.\n    match = make_match_prompt(row, "테스트 전사")\n    assert "테스트 전사" in match and match.endswith(MATCH_INSTRUCTION)\n    body = lambda s: s.split("질문: ", 1)[1].rsplit(chr(10), 1)[0]\n    assert body(match) == body(make_prompt(row)), "2단계 질문/선지 본문이 E1과 다릅니다"\n    print("공통 프롬프트/파서 OK (엄격 + CoT + E3 2단계)")\n\n\nif __name__ == "__main__":\n    _self_test()\n', 'vlm_runtime.py': '"""E4 이후 실험이 공유하는 Qwen3.5 VLM 적재·추론·짝 비교 코드.\n\nE1~E3는 이 코드를 노트북마다 복사했습니다. E4(27B 제로샷)와 E5(QLoRA)는 E1과\n문항 단위로 맞대야 하므로 적재·메시지·생성 규칙을 여기 한 벌만 둡니다.\n프롬프트와 파서는 여전히 common_eval에서 옵니다.\n"""\nimport json\nimport math\nimport time\nfrom math import comb\nfrom pathlib import Path\n\nimport pandas as pd\nimport torch\n\nimport common_eval as ce\n\nE1_PRED = Path("output/exp_E1_qwen35_9b/predictions.jsonl")\nE1_TAXONOMY = Path("output/exp_E1_qwen35_9b/error_taxonomy.csv")\n\n\ndef check_weights(model_dir):\n    """config와 index에 적힌 shard가 전부 있는지 확인합니다. 다운로드 중단을 잡습니다."""\n    model_dir = Path(model_dir)\n    assert (model_dir / "config.json").is_file(), f"로컬 모델을 준비하세요: {model_dir}"\n    index = model_dir / "model.safetensors.index.json"\n    if index.is_file():\n        need = sorted(set(json.loads(index.read_text())["weight_map"].values()))\n        miss = [f for f in need if not (model_dir / f).is_file()]\n        assert not miss, f"가중치 shard {len(miss)}/{len(need)}개 누락: {miss[:2]}"\n    assert list(model_dir.glob("*.safetensors")), f"가중치(.safetensors)가 없습니다: {model_dir}"\n\n\ndef build_device_map(model_dir, cpu_modules=(), cpu_last_layers=0):\n    """GPU에 다 못 올릴 때 CPU로 보낼 모듈을 명시합니다.\n\n    device_map="auto"는 앞에서부터 채우므로 비전 인코더나 앞쪽 층이 CPU로 밀릴 수 있습니다.\n    27B는 임베딩과 lm_head(각 약 2.5 GiB, 양자화 대상 아님)만 빼면 4bit 본체가 16 GiB에 들어가므로\n    그 둘을 먼저 내리고, 그래도 부족할 때만 마지막 층부터 내립니다.\n    """\n    cfg = json.loads((Path(model_dir) / "config.json").read_text(encoding="utf-8"))\n    n_layers = cfg["text_config"]["num_hidden_layers"]\n    lm = "model.language_model"\n    dmap = {"model.visual": 0, f"{lm}.embed_tokens": 0, f"{lm}.norm": 0,\n            f"{lm}.rotary_emb": 0, "lm_head": 0}\n    for i in range(n_layers):\n        dmap[f"{lm}.layers.{i}"] = "cpu" if i >= n_layers - cpu_last_layers else 0\n    for name in cpu_modules:\n        assert name in dmap, f"알 수 없는 모듈: {name}"\n        dmap[name] = "cpu"\n    return dmap\n\n\ndef execute_on_cpu(module, out_device="cuda:0"):\n    """accelerate 오프로딩 훅을 떼고 모듈을 CPU에서 직접 계산하게 바꿉니다.\n\n    기본 오프로딩은 forward마다 가중치를 GPU로 복사합니다. 27B의 임베딩·lm_head는 합쳐 5 GiB라\n    문항당 수 초가 복사에 들어갑니다. 임베딩은 조회뿐이고 lm_head는 생성 중 마지막 토큰 하나만\n    계산하므로 CPU에서 돌리고 결과만 GPU로 보내는 편이 훨씬 빠릅니다.\n    """\n    from accelerate.hooks import remove_hook_from_module\n    from accelerate.utils import set_module_tensor_to_device\n\n    hook = getattr(module, "_hf_hook", None)\n    weights = getattr(hook, "weights_map", None)\n    names = [n for n, _ in module.named_parameters()]\n    values = {n: weights[n] for n in names} if weights is not None else {}\n    remove_hook_from_module(module, recurse=True)\n    for n in names:\n        if n in values:\n            set_module_tensor_to_device(module, n, "cpu", value=values[n])\n    assert all(p.device.type == "cpu" for p in module.parameters()), "CPU 적재 실패"\n    module.register_forward_pre_hook(\n        lambda m, args, kwargs: (tuple(a.to("cpu") if torch.is_tensor(a) else a for a in args),\n                                 {k: v.to("cpu") if torch.is_tensor(v) else v\n                                  for k, v in kwargs.items()}), with_kwargs=True)\n    module.register_forward_hook(lambda m, args, out: out.to(out_device))\n\n\ndef load_vlm(model_dir, four_bit=True, dtype=torch.bfloat16, cpu_modules=(), cpu_last_layers=0,\n             adapter_dir=None):\n    """E1과 같은 nf4 설정으로 적재합니다. adapter_dir를 주면 LoRA를 얹습니다."""\n    from transformers import AutoProcessor, BitsAndBytesConfig, Qwen3_5ForConditionalGeneration\n\n    check_weights(model_dir)\n    proc = AutoProcessor.from_pretrained(str(model_dir), local_files_only=True)\n    offload = bool(cpu_modules) or cpu_last_layers > 0\n    dmap = (build_device_map(model_dir, cpu_modules, cpu_last_layers) if offload else {"": 0})\n    kwargs = dict(local_files_only=True, dtype=dtype, device_map=dmap)\n    if four_bit:\n        kwargs["quantization_config"] = BitsAndBytesConfig(\n            load_in_4bit=True, bnb_4bit_quant_type="nf4",\n            bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=dtype,\n            # CPU로 보낸 모듈은 양자화하지 않고 dtype 그대로 둡니다.\n            llm_int8_enable_fp32_cpu_offload=offload)\n    model = Qwen3_5ForConditionalGeneration.from_pretrained(str(model_dir), **kwargs)\n    for name in cpu_modules:\n        execute_on_cpu(model.get_submodule(name))\n    if adapter_dir is not None:\n        from peft import PeftModel\n        model = PeftModel.from_pretrained(model, str(adapter_dir), is_trainable=False)\n    return model.eval(), proc\n\n\ndef build_messages(row, image, consensus=False):\n    """E1과 글자 단위로 같은 메시지. 학습(E5·E7)도 이 함수로 입력을 만들어야 분포가 맞습니다.\n\n    consensus=True면 마지막 지시문만 E6의 합의 지시문으로 바뀝니다.\n    """\n    return [{"role": "system", "content": [{"type": "text", "text": ce.SYSTEM}]},\n            {"role": "user", "content": [{"type": "image", "image": image},\n                                         {"type": "text",\n                                          "text": ce.make_prompt(row, consensus=consensus)}]}]\n\n\ndef encode(proc, messages, enable_thinking=False):\n    return proc.apply_chat_template(messages, add_generation_prompt=True, tokenize=True,\n                                    return_dict=True, return_tensors="pt",\n                                    enable_thinking=enable_thinking)\n\n\ndef letter_token_ids(proc):\n    """a~d 각각이 첫 토큰이 될 수 있는 id 후보. 대소문자·앞 공백 변형을 모읍니다."""\n    tok = proc.tokenizer\n    table = {}\n    for ch in ce.CHOICES:\n        cand = set()\n        for form in (ch, " " + ch, ch.upper(), " " + ch.upper()):\n            enc = tok.encode(form, add_special_tokens=False)\n            if enc:\n                cand.add(int(enc[0]))\n        assert cand, f"\'{ch}\'의 토큰 id를 찾지 못했습니다"\n        table[ch] = sorted(cand)\n    return table\n\n\n@torch.inference_mode()\ndef answer(model, proc, inputs, limit, letters, device="cuda:0"):\n    """E1과 같은 탐욕 생성에 더해, 첫 토큰의 a~d 로그확률을 함께 돌려줍니다.\n\n    예측은 생성 결과를 parse_choice로 읽은 값 그대로이고(E1과 동일), 로그확률은\n    나중에 모델 앙상블·선지 순서 TTA에 쓰려고 추가 비용 없이 남기는 기록입니다.\n    """\n    inputs = inputs.to(device)\n    prefix_len = inputs["input_ids"].shape[-1]\n    out = model.generate(**inputs, max_new_tokens=limit, do_sample=False, use_cache=True,\n                         output_logits=True, return_dict_in_generate=True)\n    new = out.sequences[0, prefix_len:]\n    eos = model.generation_config.eos_token_id\n    eos_ids = [] if eos is None else ([eos] if isinstance(eos, int) else list(eos))\n    truncated = len(new) >= limit and (not len(new) or int(new[-1]) not in eos_ids)\n    logp = torch.log_softmax(out.logits[0][0].float(), dim=-1)\n    letter_logp = {ch: max(float(logp[i]) for i in ids) for ch, ids in letters.items()}\n    raw = proc.decode(new, skip_special_tokens=True).strip()\n    return raw, bool(truncated), int(len(new)), letter_logp\n\n\ndef run_predictions(model, proc, rows, pred_path, limit=24, resume=True, desc="predict"):\n    """문항마다 predictions.jsonl에 기록하고 flush합니다. 다시 부르면 끝난 문항을 건너뜁니다."""\n    from tqdm.auto import tqdm\n\n    pred_path = Path(pred_path)\n    done = set()\n    if resume and pred_path.is_file():\n        with pred_path.open(encoding="utf-8") as f:\n            done = {json.loads(line)["id"] for line in f if line.strip()}\n        print(f"이어서 진행: 이미 끝난 {len(done)}개 건너뜀")\n    elif not resume and pred_path.is_file():\n        pred_path.unlink()\n    todo = [r for r in rows if r["id"] not in done]\n    print(f"남은 {len(todo)}개")\n\n    letters = letter_token_ids(proc)\n    torch.cuda.reset_peak_memory_stats()\n    with pred_path.open("a", encoding="utf-8") as log:\n        for row in tqdm(todo, desc=desc):\n            torch.cuda.synchronize(); start = time.perf_counter()\n            im = ce.open_image(row)\n            inputs = encode(proc, build_messages(row, im))\n            raw, capped, n_tok, letter_logp = answer(model, proc, inputs, limit, letters)\n            torch.cuda.synchronize(); elapsed = time.perf_counter() - start\n            log.write(json.dumps(dict(\n                id=row["id"], raw_output=raw, pred=ce.parse_choice(raw), letter_logprobs=letter_logp,\n                answer_truncated=capped, answer_tokens=n_tok, seconds=elapsed,\n                image_width=im.width, image_height=im.height), ensure_ascii=False) + "\\n")\n            log.flush()\n            del inputs, im\n    return torch.cuda.max_memory_allocated() / 1024**3\n\n\ndef read_predictions(pred_path, val_df):\n    res = pd.read_json(pred_path, lines=True).drop_duplicates("id", keep="last")\n    pred_df = val_df.merge(res, on="id", validate="one_to_one", sort=False)\n    pred_df["correct"] = pred_df.pred == pred_df.answer\n    return pred_df\n\n\ndef mcnemar_exact(b, c):\n    n = b + c\n    if n == 0:\n        return 1.0\n    return min(1.0, 2 * sum(comb(n, k) for k in range(min(b, c) + 1)) / 2 ** n)\n\n\ndef paired_vs_e1(pred_df, name, e1_pred=E1_PRED, taxonomy=E1_TAXONOMY):\n    """E1과 문항 단위로 맞댑니다. 정확도 두 개를 나란히 놓는 것만으로는 부족합니다.\n\n    E1 오답 분류(error_taxonomy.csv)가 있으면 분류별로 몇 개를 고쳤는지도 봅니다.\n    27B는 판독·위치 관계를, QLoRA는 \'적힌 그대로\' 유형을 고칠 것이라는 예상을 여기서 확인합니다.\n    """\n    if not Path(e1_pred).is_file():\n        print("E1 예측이 없어 짝 비교를 건너뜁니다:", e1_pred)\n        return None\n    e1 = pd.read_json(e1_pred, lines=True).drop_duplicates("id", keep="last")\n    joined = pred_df.merge(e1[["id", "pred"]].rename(columns={"pred": "pred_e1"}), on="id")\n    ok1 = joined.pred_e1 == joined.answer\n    ok2 = joined.pred == joined.answer\n    n = len(joined)\n    b = int((ok1 & ~ok2).sum())\n    c = int((~ok1 & ok2).sum())\n    diff = (c - b) / n\n    se = math.sqrt(b + c) / n\n    p = mcnemar_exact(b, c)\n    print(f"[{n}문항]")\n    print(f"  E1   {ok1.mean():.4f} ({int(ok1.sum())}/{n})")\n    print(f"  {name} {ok2.mean():.4f} ({int(ok2.sum())}/{n})")\n    print(f"  E1만 정답 {b} / {name}만 정답 {c} / 둘 다 오답 {int((~ok1 & ~ok2).sum())}")\n    print(f"  {name} - E1 = {diff * 100:+.2f}pp  95%CI "\n          f"[{(diff - 1.96 * se) * 100:+.2f}pp, {(diff + 1.96 * se) * 100:+.2f}pp]")\n    print(f"  McNemar exact p = {p:.4g} -> "\n          + ("차이가 유의합니다" if p < 0.05 else "두 조건은 구분되지 않습니다"))\n\n    if Path(taxonomy).is_file():\n        tax = pd.read_csv(taxonomy, encoding="utf-8")\n        t = tax.merge(joined[["id", "pred", "answer"]], on="id", how="left")\n        t["fixed"] = t.pred == t.answer\n        table = t.groupby("category", sort=False).agg(E1_errors=("id", "size"),\n                                                      fixed=("fixed", "sum"))\n        print("\\nE1 오답 분류별로 고친 수 (label_error·ambiguous는 고쳐도 운에 가깝습니다)")\n        print(table.to_string())\n    return dict(n=n, e1_only=b, new_only=c, diff=diff, p=p)\n\n\ndef _self_test():\n    assert mcnemar_exact(0, 0) == 1.0\n    assert mcnemar_exact(11, 11) == 1.0\n    assert abs(mcnemar_exact(11, 17) - 0.3449) < 1e-3   # E2 vs E1 기록값\n    row = {"question": "Q", "a": "A", "b": "B", "c": "C", "d": "D"}\n    assert build_messages(row, None)[1]["content"][1]["text"] == ce.make_prompt(row)\n    print("vlm_runtime OK")\n\n\nif __name__ == "__main__":\n    _self_test()\n', 'qlora.py': '"""E5 QLoRA 학습 코드.\n\n핵심 규칙은 하나입니다. **학습 입력은 추론 입력과 토큰 단위로 같아야 합니다.**\n그래서 프롬프트는 vlm_runtime.build_messages + encode(add_generation_prompt=True)로 만들고,\n그 뒤에 정답 글자와 <|im_end|>만 붙입니다. 채팅 템플릿에 assistant 메시지를 넣어 렌더링하면\nthinking 블록 처리가 추론 때와 달라질 수 있어 쓰지 않습니다.\n"""\nimport json\nimport math\nfrom pathlib import Path\n\nimport pandas as pd\nimport torch\n\nimport common_eval as ce\nimport vlm_runtime as vr\n\n# 언어 모델 쪽 선형층만. 비전 인코더와 merger는 얼립니다(E1 오답의 다수가 \'읽고 나서 고치는\' 쪽이라\n# 우선 언어 쪽 습관을 바꿔 보고, 판독 실패가 안 줄면 그때 merger를 엽니다).\nLORA_TARGET = (r"model\\.language_model\\.layers\\.\\d+\\."\n               r"(self_attn\\.(q_proj|k_proj|v_proj|o_proj)"\n               r"|linear_attn\\.(in_proj_qkv|in_proj_z|out_proj)"\n               r"|mlp\\.(gate_proj|up_proj|down_proj))")\n\n\ndef load_train_rows(val_df, train_csv=ce.DATA_DIR / "train.csv", seed=42):\n    """train.csv에서 검증 672문항을 뺀 나머지 전부. 검증과 ID·이미지가 겹치지 않는지 확인합니다."""\n    train = pd.read_csv(train_csv, keep_default_na=False)\n    val_ids, val_paths = set(val_df.id), set(val_df.path)\n    fit = train[~train.id.isin(val_ids)].copy()\n    assert not set(fit.path) & val_paths, "학습 이미지가 검증과 겹칩니다"\n    assert len(fit) + len(val_ids) == len(train), "검증 ID 일부가 train.csv에 없습니다"\n    assert set(fit.answer) <= set(ce.CHOICES), "정답 형식이 a~d가 아닙니다"\n    return fit.sample(frac=1.0, random_state=seed).reset_index(drop=True)\n\n\nDEV_VOTES = [f"answer{i}" for i in range(1, 6)]\n\n\ndef dev_majority(row, min_votes=3):\n    """dev의 5명 응답 중 min_votes표 이상 받은 단독 최다 답. 없으면 None."""\n    votes = [row[c] for c in DEV_VOTES if row.get(c) in ce.CHOICES]\n    if not votes:\n        return None\n    counts = {ch: votes.count(ch) for ch in set(votes)}\n    best = max(counts.values())\n    leaders = [ch for ch, k in counts.items() if k == best]\n    return leaders[0] if best >= min_votes and len(leaders) == 1 else None\n\n\ndef rows_from_manifest(manifest, train_df, dev_df, holdout_ids, min_votes=3):\n    """학습 구성 목록(id, source)을 학습 행으로 바꿉니다. 정답 글자는 저장소에 두지 않고 여기서 붙입니다.\n\n    train 문항은 train.csv의 정답을, dev 문항은 다수결(min_votes표 이상, 단독 최다)을 씁니다.\n    holdout 문항이 섞이면 평가가 무의미해지므로 멈춥니다.\n    """\n    assert set(manifest.source) <= {"train", "dev"}, f"알 수 없는 source: {set(manifest.source)}"\n    leaked = set(manifest.id) & set(holdout_ids)\n    assert not leaked, f"holdout 문항 {len(leaked)}개가 학습 구성에 들어 있습니다: {sorted(leaked)[:3]}"\n    tr = train_df.set_index("id")\n    dv = dev_df.set_index("id")\n    rows, dropped = [], 0\n    for item in manifest.itertuples():\n        if item.source == "train":\n            rows.append({"id": item.id, **tr.loc[item.id].to_dict()})\n        else:\n            r = dv.loc[item.id].to_dict()\n            answer = dev_majority(r, min_votes)\n            if answer is None:\n                dropped += 1\n                continue\n            rows.append({"id": item.id, **{k: v for k, v in r.items() if k not in DEV_VOTES},\n                         "answer": answer})\n    if dropped:\n        print(f"dev 다수결이 {min_votes}표에 못 미치거나 동률이라 뺀 문항 {dropped}개")\n    return rows\n\n\ndef target_ids(proc, letter):\n    tok = proc.tokenizer\n    ids = tok.encode(letter, add_special_tokens=False)\n    assert len(ids) == 1, f"\'{letter}\'가 한 토큰이 아닙니다: {ids}"\n    im_end = tok.convert_tokens_to_ids("<|im_end|>")\n    assert isinstance(im_end, int) and im_end >= 0\n    return ids + [im_end]\n\n\ndef make_example(proc, row, image, consensus=False):\n    """추론과 같은 프롬프트 + [정답 글자, <|im_end|>]. 손실은 뒤의 두 토큰에만 겁니다."""\n    inputs = vr.encode(proc, vr.build_messages(row, image, consensus=consensus))\n    tgt = torch.tensor([target_ids(proc, row["answer"])], dtype=inputs["input_ids"].dtype)\n    n_prompt = inputs["input_ids"].shape[-1]\n    out = {}\n    for k, v in inputs.items():\n        if torch.is_tensor(v) and v.dim() == 2 and v.shape == (1, n_prompt):\n            fill = torch.ones_like(tgt) if k == "attention_mask" else (\n                tgt if k == "input_ids" else torch.zeros_like(tgt))\n            out[k] = torch.cat([v, fill.to(v.dtype)], dim=1)\n        else:\n            out[k] = v\n    return out, tgt\n\n\ndef answer_loss(model, batch, tgt, device="cuda:0"):\n    """정답 두 토큰에 대한 CE. 마지막 n+1개 위치의 로짓만 계산해 248K 어휘 전체 로짓을 피합니다.\n\n    위치 p의 로짓은 p+1 토큰을 예측하므로, 끝에서 n+1개를 남기면 앞의 n개가 정답 토큰을 예측합니다.\n    """\n    batch = {k: (v.to(device) if torch.is_tensor(v) else v) for k, v in batch.items()}\n    tgt = tgt.to(device)\n    n = tgt.shape[-1]\n    logits = model(**batch, logits_to_keep=n + 1, use_cache=False).logits[:, :n].float()\n    return torch.nn.functional.cross_entropy(logits.reshape(-1, logits.shape[-1]), tgt.reshape(-1))\n\n\ndef attach_lora(model, r=16, alpha=32, dropout=0.05):\n    from peft import LoraConfig, get_peft_model\n\n    for p in model.parameters():\n        p.requires_grad_(False)\n    model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})\n    model.enable_input_require_grads()\n    cfg = LoraConfig(r=r, lora_alpha=alpha, lora_dropout=dropout, bias="none",\n                     target_modules=LORA_TARGET, task_type="CAUSAL_LM")\n    model = get_peft_model(model, cfg)\n    n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)\n    assert n_train > 0, "LoRA가 붙은 층이 없습니다. LORA_TARGET을 확인하세요"\n    return model\n\n\ndef cosine_lr(step, total, warmup):\n    if step < warmup:\n        return (step + 1) / warmup\n    progress = (step - warmup) / max(1, total - warmup)\n    return 0.5 * (1 + math.cos(math.pi * min(1.0, progress)))\n\n\ndef save_checkpoint(model, optimizer, scheduler, state, ckpt_dir):\n    """어댑터 + 옵티마이저 + 진행 위치. Windows 장시간 학습이 끊겨도 이어갈 수 있게 합니다."""\n    ckpt_dir = Path(ckpt_dir)\n    tmp = ckpt_dir.with_name(ckpt_dir.name + ".tmp")\n    model.save_pretrained(str(tmp))\n    torch.save({"optimizer": optimizer.state_dict(), "scheduler": scheduler.state_dict()},\n               tmp / "train_state.pt")\n    (tmp / "progress.json").write_text(json.dumps(state, indent=2), encoding="utf-8")\n    if ckpt_dir.exists():\n        import shutil\n        shutil.rmtree(ckpt_dir)\n    tmp.rename(ckpt_dir)\n\n\ndef load_checkpoint(model, optimizer, scheduler, ckpt_dir):\n    from peft import set_peft_model_state_dict\n    from safetensors.torch import load_file\n\n    ckpt_dir = Path(ckpt_dir)\n    set_peft_model_state_dict(model, load_file(str(ckpt_dir / "adapter_model.safetensors")))\n    st = torch.load(ckpt_dir / "train_state.pt", map_location="cpu", weights_only=False)\n    optimizer.load_state_dict(st["optimizer"])\n    scheduler.load_state_dict(st["scheduler"])\n    return json.loads((ckpt_dir / "progress.json").read_text(encoding="utf-8"))\n\n\ndef train_loop(model, proc, order, *, grad_accum, lr, warmup_ratio, max_grad_norm, save_every,\n               ckpt_dir, log_path, consensus=False, resume=True):\n    """문항 하나씩 손실을 쌓아 grad_accum마다 갱신합니다. save_every 스텝마다 ckpt_dir에 저장하고,\n    다시 부르면 마지막 체크포인트에서 이어 갑니다. Colab 세션이 끊겨도 Drive의 ckpt_dir로 재개합니다.\n    """\n    import time\n    import common_eval as _ce\n\n    total = len(order) // grad_accum\n    warmup = max(1, int(total * warmup_ratio))\n    params = [p for p in model.parameters() if p.requires_grad]\n    optimizer = torch.optim.AdamW(params, lr=lr, weight_decay=0.0)\n    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lambda s: cosine_lr(s, total, warmup))\n    ckpt_dir, log_path = Path(ckpt_dir), Path(log_path)\n    start = 0\n    if resume and (ckpt_dir / "progress.json").is_file():\n        start = load_checkpoint(model, optimizer, scheduler, ckpt_dir)["step"]\n        print(f"체크포인트에서 이어서: {start}/{total} 스텝")\n    elif log_path.is_file():\n        log_path.unlink()\n    model.train()\n    t_start = time.time()\n    with log_path.open("a", encoding="utf-8") as log:\n        for step in range(start, total):\n            t0 = time.time()\n            losses = []\n            for row in order[step * grad_accum:(step + 1) * grad_accum]:\n                batch, tgt = make_example(proc, row, _ce.open_image(row), consensus=consensus)\n                loss = answer_loss(model, batch, tgt)\n                (loss / grad_accum).backward()\n                losses.append(loss.item())\n                del batch, tgt, loss\n            grad_norm = torch.nn.utils.clip_grad_norm_(params, max_grad_norm)\n            optimizer.step(); scheduler.step(); optimizer.zero_grad(set_to_none=True)\n            item = dict(step=step + 1, loss=sum(losses) / len(losses), max_loss=max(losses),\n                        lr=scheduler.get_last_lr()[0], grad_norm=float(grad_norm),\n                        seconds=time.time() - t0)\n            log.write(json.dumps(item) + "\\n"); log.flush()\n            done = step + 1 - start\n            if done == 1 or (step + 1) % 10 == 0 or step + 1 == total:\n                left = (time.time() - t_start) / done * (total - step - 1)\n                print(f"  {step + 1}/{total}  loss {item[\'loss\']:.4f}  max {item[\'max_loss\']:.3f}  "\n                      f"{item[\'seconds\']:.0f}초/스텝  남은 약 {left / 60:.0f}분", flush=True)\n            if (step + 1) % save_every == 0 or step + 1 == total:\n                save_checkpoint(model, optimizer, scheduler, {"step": step + 1, "total": total}, ckpt_dir)\n    return total\n\n\ndef merge_and_save(base_dir, adapter_dir, out_dir, dtype=torch.bfloat16, device="cuda:0"):\n    """원본(bf16)에 어댑터를 합쳐 일반 체크포인트로 저장합니다. vLLM은 이 폴더를 그대로 올립니다.\n\n    vLLM이 Qwen3.5 linear attention 층의 LoRA를 직접 받는지 보장할 수 없어 합친 모델로 평가합니다.\n    4bit로 학습했어도 합치는 대상은 bf16 원본입니다.\n    """\n    from peft import PeftModel\n    from transformers import AutoProcessor, Qwen3_5ForConditionalGeneration\n\n    base = Qwen3_5ForConditionalGeneration.from_pretrained(\n        str(base_dir), dtype=dtype, device_map={"": device}, local_files_only=True)\n    merged = PeftModel.from_pretrained(base, str(adapter_dir)).merge_and_unload()\n    out_dir = Path(out_dir)\n    merged.save_pretrained(str(out_dir), safe_serialization=True, max_shard_size="5GB")\n    AutoProcessor.from_pretrained(str(base_dir), local_files_only=True).save_pretrained(str(out_dir))\n    del merged, base\n    copy_missing_weights(base_dir, out_dir)\n    return out_dir\n\n\ndef copy_missing_weights(base_dir, out_dir):\n    """원본에는 있는데 저장본에 없는 가중치(mtp.* 다중 토큰 예측 층)를 그대로 옮겨 담습니다.\n\n    transformers는 mtp 층을 불러오지 않아 save_pretrained 결과에서 빠집니다. 학습과 무관한 층이지만\n    config에는 남아 있어, vLLM이 찾을 경우를 대비해 원본과 같은 가중치 집합으로 맞춥니다.\n    """\n    from safetensors import safe_open\n    from safetensors.torch import save_file\n\n    base_dir, out_dir = Path(base_dir), Path(out_dir)\n    base_map = json.loads((base_dir / "model.safetensors.index.json").read_text())["weight_map"]\n    index_path = out_dir / "model.safetensors.index.json"\n    index = json.loads(index_path.read_text())\n    missing = sorted(set(base_map) - set(index["weight_map"]))\n    assert all(k.startswith("mtp.") for k in missing), f"예상 밖 누락: {[k for k in missing if not k.startswith(\'mtp.\')][:3]}"\n    if not missing:\n        return []\n    tensors = {}\n    for shard in sorted({base_map[k] for k in missing}):\n        with safe_open(str(base_dir / shard), framework="pt") as f:\n            for k in missing:\n                if base_map[k] == shard:\n                    tensors[k] = f.get_tensor(k)\n    save_file(tensors, str(out_dir / "model-extra-mtp.safetensors"), metadata={"format": "pt"})\n    for k in missing:\n        index["weight_map"][k] = "model-extra-mtp.safetensors"\n    index_path.write_text(json.dumps(index, indent=2), encoding="utf-8")\n    return missing\n\n\ndef _self_test():\n    import re\n    ok = "model.language_model.layers.3.linear_attn.in_proj_qkv"\n    assert re.fullmatch(LORA_TARGET, ok)\n    assert re.fullmatch(LORA_TARGET, "model.language_model.layers.7.mlp.down_proj")\n    assert not re.fullmatch(LORA_TARGET, "model.visual.blocks.0.attn.qkv")\n    assert not re.fullmatch(LORA_TARGET, "lm_head")\n    assert abs(cosine_lr(0, 100, 10) - 0.1) < 1e-9 and abs(cosine_lr(100, 100, 10)) < 1e-9\n    votes = lambda *v: dict(zip(DEV_VOTES, list(v) + [""] * (5 - len(v))))\n    assert dev_majority(votes("a", "a", "a", "b", "c")) == "a"\n    assert dev_majority(votes("a", "a", "b", "b", "c")) is None        # 2표 동률\n    assert dev_majority(votes("a", "a", "b", "c", "d")) is None        # 2표 단독이지만 3표 미만\n    assert dev_majority(votes("a", "a", "b", "c", "d"), min_votes=2) == "a"\n    assert dev_majority(votes()) is None\n    print("qlora OK")\n\n\nif __name__ == "__main__":\n    _self_test()\n', 'vllm_eval.py': '"""Colab vLLM 추론 공통 코드 (E7 이후).\n\nE4·E6 Colab 노트북은 이 코드를 셀에 복사해 두었습니다. E7부터는 노트북이 이 파일 원문을 품고\nimport합니다. 메시지·이미지 규칙·파서는 로컬 추론(vlm_runtime)과 같아야 하므로 common_eval을 씁니다.\n"""\nimport json\nimport math\nimport shutil\nimport time\nfrom math import comb\nfrom pathlib import Path\n\nimport pandas as pd\n\nimport common_eval as ce\n\nCHUNK = 256\n\n\ndef letter_token_ids(tokenizer):\n    return {ch: sorted({tokenizer.encode(f, add_special_tokens=False)[0]\n                        for f in (ch, " " + ch, ch.upper(), " " + ch.upper())\n                        if tokenizer.encode(f, add_special_tokens=False)})\n            for ch in ce.CHOICES}\n\n\ndef make_request(proc, row, consensus=False):\n    """로컬과 같은 메시지를 HF 채팅 템플릿으로 렌더링하고, 이미지는 같은 open_image로 줄여 넘깁니다."""\n    image = ce.open_image(row)\n    messages = [{"role": "system", "content": [{"type": "text", "text": ce.SYSTEM}]},\n                {"role": "user", "content": [{"type": "image"},\n                                             {"type": "text",\n                                              "text": ce.make_prompt(row, consensus=consensus)}]}]\n    prompt = proc.apply_chat_template(messages, tokenize=False, add_generation_prompt=True,\n                                      enable_thinking=False)\n    return {"prompt": prompt, "multi_modal_data": {"image": image}}, image.size\n\n\ndef letter_logprobs(first_step, letters):\n    """상위 k개 안에 없는 글자는 -1e9. JSON에 -inf를 쓰지 않으려고 유한값을 씁니다."""\n    out = {}\n    for ch, ids in letters.items():\n        vals = [first_step[i].logprob for i in ids if i in first_step]\n        out[ch] = max(vals) if vals else -1e9\n    return out\n\n\ndef run(llm, proc, sampling, rows, out_dir, drive_dir=None, consensus=False, desc=""):\n    """묶음 단위로 추론해 predictions.jsonl에 쓰고, 묶음마다 drive_dir로 복사합니다.\n\n    다시 부르면 기록된 문항을 건너뜁니다. out_dir에 기록이 없고 drive_dir에 있으면 먼저 가져옵니다.\n    """\n    out_dir = Path(out_dir)\n    out_dir.mkdir(parents=True, exist_ok=True)\n    if drive_dir is not None:\n        drive_dir = Path(drive_dir)\n        drive_dir.mkdir(parents=True, exist_ok=True)\n    pred_path = out_dir / "predictions.jsonl"\n    if not pred_path.is_file() and drive_dir is not None and (drive_dir / "predictions.jsonl").is_file():\n        shutil.copy2(drive_dir / "predictions.jsonl", pred_path)\n        print("Drive에서 이전 기록을 복구했습니다")\n    done = set()\n    if pred_path.is_file():\n        done = {json.loads(l)["id"] for l in pred_path.open(encoding="utf-8") if l.strip()}\n    todo = [r for r in rows if r["id"] not in done]\n    print(f"[{desc or out_dir.name}] 대상 {len(rows)} / 기록됨 {len(done)} / 남은 {len(todo)}")\n\n    letters = letter_token_ids(proc.tokenizer)\n    instruction = ce.CONSENSUS_INSTRUCTION if consensus else ce.ANSWER_INSTRUCTION\n    t_all = time.time()\n    for start in range(0, len(todo), CHUNK):\n        batch = todo[start:start + CHUNK]\n        t0 = time.time()\n        reqs, sizes = zip(*(make_request(proc, r, consensus) for r in batch))\n        if start == 0:\n            assert "<|image_pad|>" in reqs[0]["prompt"], "프롬프트에 이미지 토큰이 없습니다"\n            assert reqs[0]["prompt"].count(instruction) == 1, "지시문이 기대와 다릅니다"\n        outs = llm.generate(list(reqs), sampling, use_tqdm=False)\n        elapsed = time.time() - t0\n        with pred_path.open("a", encoding="utf-8") as log:\n            for row, size, o in zip(batch, sizes, outs):\n                c = o.outputs[0]\n                raw = c.text.strip()\n                log.write(json.dumps(dict(\n                    id=row["id"], raw_output=raw, pred=ce.parse_choice(raw),\n                    letter_logprobs=letter_logprobs(c.logprobs[0], letters) if c.logprobs else {},\n                    answer_truncated=c.finish_reason == "length", answer_tokens=len(c.token_ids),\n                    seconds=elapsed / len(batch), image_width=size[0], image_height=size[1]),\n                    ensure_ascii=False) + "\\n")\n        if drive_dir is not None:\n            for f in out_dir.glob("*"):\n                if f.is_file():\n                    shutil.copy2(f, drive_dir / f.name)\n        n_done = start + len(batch)\n        print(f"  {n_done}/{len(todo)}  {elapsed / len(batch):.3f}초/문항  "\n              f"남은 약 {(time.time() - t_all) / n_done * (len(todo) - n_done) / 60:.0f}분")\n    return read(pred_path)\n\n\ndef read(pred_path):\n    res = pd.read_json(pred_path, lines=True).drop_duplicates("id", keep="last")\n    res["conf"] = res.letter_logprobs.map(\n        lambda lp: math.exp(max(lp.values())) if isinstance(lp, dict) and lp else float("nan"))\n    return res\n\n\ndef fill_invalid(res):\n    """제출용. 파서가 실패한 문항은 기록된 a~d 로그확률 중 가장 높은 글자로 채웁니다."""\n    res = res.copy()\n    bad = res.pred == "INVALID"\n    res.loc[bad, "pred"] = res.loc[bad, "letter_logprobs"].map(\n        lambda lp: max(lp, key=lp.get) if lp else "INVALID")\n    return res, int(bad.sum())\n\n\ndef mcnemar_exact(b, c):\n    n = b + c\n    return 1.0 if n == 0 else min(1.0, 2 * sum(comb(n, k) for k in range(min(b, c) + 1)) / 2 ** n)\n\n\ndef paired(gold, pred_a, pred_b, name_a, name_b):\n    """gold(id, answer)에 두 예측(id, pred)을 맞대 McNemar를 출력합니다."""\n    j = (gold[["id", "answer"]]\n         .merge(pred_a[["id", "pred"]].rename(columns={"pred": "a"}), on="id")\n         .merge(pred_b[["id", "pred"]].rename(columns={"pred": "b"}), on="id"))\n    ok_a, ok_b = j.a == j.answer, j.b == j.answer\n    b, c = int((ok_a & ~ok_b).sum()), int((~ok_a & ok_b).sum())\n    n = len(j)\n    diff, se = (c - b) / n, math.sqrt(b + c) / n\n    p = mcnemar_exact(b, c)\n    print(f"[{n}문항] {name_a} {ok_a.mean():.4f} / {name_b} {ok_b.mean():.4f}")\n    print(f"  {name_a}만 정답 {b} / {name_b}만 정답 {c} / 차이 {diff * 100:+.2f}pp "\n          f"95%CI [{(diff - 1.96 * se) * 100:+.2f}, {(diff + 1.96 * se) * 100:+.2f}] / McNemar p = {p:.4g}")\n    return dict(n=n, a_only=b, b_only=c, diff=diff, p=p)\n\n\ndef _self_test():\n    assert abs(mcnemar_exact(11, 17) - 0.3449) < 1e-3\n    res = pd.DataFrame({"id": ["x", "y"], "pred": ["INVALID", "b"],\n                        "letter_logprobs": [{"a": -3.0, "b": -2.0, "c": -0.1, "d": -5.0}, {}]})\n    filled, n = fill_invalid(res)\n    assert n == 1 and list(filled.pred) == ["c", "b"]\n    print("vllm_eval OK")\n\n\nif __name__ == "__main__":\n    _self_test()\n'}
SOURCE_SHA = {'common_eval.py': 'ff6dc88f6916fa149a3a8fdec7dfa60fa384701c542aa19fedc582ce6f8850ae', 'vlm_runtime.py': 'e6735a19ea96a3c768bba136c8c36de7ec76efaa9494c0c8b6c955e97555bfd0', 'qlora.py': '56f954f09db990bebadb03b7165c5227bef34e257add3c7f9d17dc5100e16a59', 'vllm_eval.py': '85c125c2caeed06a7da1e82c3776ff469240fcc9720497fc6ea2abd1970ec70e'}
import hashlib
SRC_DIR = Path("/content/src"); SRC_DIR.mkdir(parents=True, exist_ok=True)
for name, text in SOURCES.items():
    assert hashlib.sha256(text.encode("utf-8")).hexdigest() == SOURCE_SHA[name], name
    (SRC_DIR / name).write_text(text, encoding="utf-8")
sys.path.insert(0, str(SRC_DIR))
import common_eval as ce
import vlm_runtime as vr
import qlora
import vllm_eval as ve

ce.DATA_DIR = DATA_ROOT
ce._self_test(); vr._self_test(); qlora._self_test(); ve._self_test()
print("데이터:", DATA_ROOT)

## 3. 설정

`QUANT="bf16"`이 기본입니다. A100 80GB에 27B bf16(약 54 GiB)이 들어가고, 4bit보다 학습이 빠르며
제출 추론(vLLM bf16)과 조건이 같습니다. 메모리가 부족하면 `"nf4"`(QLoRA)로 바꾸세요.

`CONSENSUS`는 E6 3단계에서 합의 지시문이 holdout에서 유의하게 이겼을 때만 `True`로 두세요.
학습과 평가가 같은 지시문을 씁니다.

In [ ]:
CONFIG = "B"
QUANT = "bf16"               # "bf16"(LoRA) 또는 "nf4"(QLoRA)
CONSENSUS = False            # E6에서 합의 지시문이 이겼을 때만 True
LORA_R, LORA_ALPHA, LORA_DROPOUT = 16, 32, 0.05
LR = 1e-4
GRAD_ACCUM = 16
EPOCHS = 1
WARMUP_RATIO = 0.03
MAX_GRAD_NORM = 1.0
SAVE_EVERY = 20              # 옵티마이저 스텝. 20스텝 = 약 320문항
MAX_TRAIN_ROWS = None        # int면 앞 N문항만 학습(배관 점검용). 결과를 비교에 쓰지 마세요
MAX_NEW_TOKENS = 24

RUN_NAME = f"exp_E7_27b_{CONFIG}" + ("_cons" if CONSENSUS else "") + ("_smoke" if MAX_TRAIN_ROWS else "")
DRIVE_RUN = DRIVE_BASE / "output" / RUN_NAME
CKPT_DIR = DRIVE_RUN / "checkpoint"
ADAPTER_DIR = DRIVE_RUN / "adapter"
TRAIN_LOG = DRIVE_RUN / "train_log.jsonl"
BASE_DIR = Path("/content/models/Qwen3.5-27B")
MERGED_DIR = Path("/content/merged") / RUN_NAME
DRIVE_RUN.mkdir(parents=True, exist_ok=True)
DTYPE = torch.bfloat16
torch.manual_seed(42)
print("실행 이름:", RUN_NAME, "/ Drive:", DRIVE_RUN)

## 4. 학습 데이터

구성 목록(`splits/e7_train_B.csv`, SHA-256 `e61fdad30c012324…`)에는 id만 있고,
정답은 train.csv(train 문항)와 dev.csv 다수결(dev 문항, 3표 이상 단독 최다)로 붙입니다.
holdout 문항이 하나라도 섞이면 멈춥니다.

In [ ]:
TRAIN_NUMS = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 58, 59, 60, 62, 63, 64, 66, 67, 68, 69, 70, 71, 73, 74, 75, 76, 79, 80, 81, 82, 83, 85, 86, 87, 88, 89, 90, 92, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 107, 109, 112, 114, 115, 116, 118, 120, 121, 122, 123, 125, 126, 128, 129, 131, 133, 135, 136, 137, 138, 139, 140, 141, 142, 144, 146, 147, 148, 149, 150, 151, 152, 153, 154, 157, 158, 159, 160, 161, 162, 163, 164, 165, 167, 168, 170, 171, 172, 173, 176, 178, 179, 180, 181, 182, 183, 184, 186, 187, 188, 189, 190, 191, 194, 195, 196, 197, 198, 199, 200, 201, 202, 204, 206, 207, 210, 211, 212, 215, 216, 217, 218, 219, 220, 222, 223, 227, 228, 229, 232, 233, 235, 236, 237, 240, 242, 243, 244, 246, 248, 249, 250, 253, 254, 255, 256, 257, 258, 259, 261, 262, 263, 264, 265, 268, 269, 270, 272, 274, 275, 276, 278, 279, 282, 283, 284, 285, 286, 287, 288, 289, 291, 292, 293, 294, 295, 296, 297, 298, 301, 302, 305, 306, 308, 310, 311, 313, 314, 315, 318, 320, 321, 324, 326, 327, 328, 329, 330, 331, 332, 333, 334, 336, 337, 338, 340, 341, 342, 343, 344, 345, 348, 349, 350, 351, 352, 353, 354, 357, 358, 359, 361, 362, 364, 366, 367, 368, 369, 372, 373, 374, 375, 376, 377, 378, 379, 380, 382, 383, 384, 385, 386, 389, 391, 392, 393, 394, 395, 396, 397, 398, 399, 400, 402, 403, 404, 405, 406, 407, 408, 409, 410, 411, 412, 413, 414, 415, 416, 417, 418, 419, 420, 421, 422, 423, 424, 425, 426, 427, 429, 430, 431, 433, 434, 435, 436, 437, 438, 439, 440, 441, 443, 444, 445, 446, 447, 448, 449, 450, 451, 452, 453, 454, 455, 456, 457, 458, 459, 460, 461, 462, 464, 465, 466, 467, 468, 469, 470, 471, 472, 473, 474, 476, 477, 478, 479, 482, 483, 485, 486, 487, 488, 489, 490, 491, 492, 493, 494, 495, 496, 497, 498, 499, 501, 502, 503, 504, 506, 507, 509, 510, 511, 512, 513, 514, 516, 518, 520, 521, 522, 523, 525, 526, 527, 530, 532, 533, 534, 535, 536, 538, 539, 540, 541, 543, 544, 545, 546, 547, 548, 549, 550, 552, 553, 554, 555, 556, 557, 559, 560, 561, 562, 563, 564, 565, 566, 568, 571, 572, 576, 577, 578, 579, 580, 581, 582, 583, 585, 586, 587, 588, 589, 590, 591, 592, 593, 595, 596, 597, 598, 600, 602, 604, 605, 606, 607, 608, 609, 610, 611, 612, 615, 616, 617, 618, 619, 620, 621, 622, 623, 624, 626, 627, 628, 629, 630, 631, 632, 633, 634, 635, 636, 637, 638, 640, 642, 643, 644, 645, 646, 647, 648, 651, 653, 654, 655, 656, 657, 658, 659, 660, 661, 662, 663, 664, 665, 666, 667, 668, 669, 671, 672, 674, 676, 677, 678, 679, 680, 681, 682, 683, 684, 685, 686, 687, 688, 689, 690, 691, 692, 693, 694, 695, 696, 697, 699, 700, 701, 702, 704, 705, 706, 707, 708, 709, 710, 711, 712, 713, 714, 715, 716, 717, 719, 720, 721, 722, 724, 725, 726, 727, 728, 729, 730, 731, 732, 733, 734, 735, 736, 737, 738, 739, 740, 741, 743, 744, 745, 747, 748, 749, 750, 751, 752, 753, 754, 755, 757, 759, 760, 761, 762, 763, 764, 766, 767, 768, 769, 770, 771, 772, 773, 774, 777, 779, 780, 781, 782, 783, 784, 788, 789, 790, 791, 793, 794, 795, 796, 798, 799, 800, 801, 802, 804, 805, 806, 808, 809, 810, 811, 812, 813, 814, 815, 816, 817, 818, 819, 820, 821, 822, 823, 824, 825, 826, 828, 830, 831, 832, 834, 835, 836, 837, 838, 839, 840, 841, 843, 844, 845, 847, 848, 850, 851, 852, 853, 854, 855, 856, 857, 859, 860, 861, 862, 864, 865, 866, 867, 868, 869, 872, 874, 876, 877, 878, 879, 880, 881, 883, 884, 885, 886, 887, 888, 889, 890, 891, 892, 894, 895, 896, 897, 898, 899, 900, 901, 902, 903, 904, 905, 906, 907, 908, 910, 911, 914, 915, 916, 917, 918, 919, 920, 921, 922, 923, 924, 925, 926, 927, 928, 929, 930, 931, 932, 933, 934, 935, 936, 938, 941, 942, 943, 945, 946, 947, 948, 950, 951, 952, 953, 954, 955, 956, 957, 958, 959, 960, 961, 962, 963, 964, 965, 966, 967, 969, 970, 971, 972, 973, 974, 975, 976, 977, 978, 980, 982, 984, 986, 987, 988, 989, 990, 991, 992, 993, 994, 995, 996, 997, 998, 999, 1001, 1002, 1004, 1007, 1008, 1009, 1010, 1011, 1013, 1014, 1015, 1016, 1017, 1019, 1020, 1021, 1023, 1024, 1025, 1026, 1027, 1028, 1029, 1030, 1031, 1032, 1035, 1036, 1037, 1038, 1040, 1041, 1043, 1044, 1046, 1047, 1050, 1051, 1053, 1054, 1055, 1056, 1057, 1058, 1059, 1060, 1062, 1063, 1064, 1066, 1067, 1068, 1069, 1071, 1073, 1074, 1076, 1077, 1078, 1081, 1082, 1084, 1085, 1086, 1088, 1089, 1090, 1091, 1093, 1095, 1096, 1098, 1099, 1100, 1101, 1102, 1103, 1104, 1105, 1107, 1108, 1109, 1110, 1111, 1112, 1113, 1114, 1115, 1121, 1122, 1124, 1125, 1126, 1128, 1129, 1130, 1131, 1132, 1133, 1135, 1136, 1137, 1138, 1140, 1141, 1142, 1143, 1144, 1146, 1147, 1148, 1151, 1153, 1154, 1157, 1158, 1159, 1160, 1161, 1162, 1163, 1164, 1165, 1167, 1168, 1169, 1170, 1173, 1175, 1176, 1177, 1178, 1179, 1182, 1183, 1185, 1186, 1187, 1188, 1189, 1190, 1191, 1192, 1194, 1195, 1196, 1198, 1202, 1203, 1204, 1205, 1206, 1207, 1208, 1209, 1210, 1212, 1214, 1215, 1216, 1217, 1219, 1220, 1222, 1225, 1226, 1227, 1228, 1229, 1231, 1232, 1233, 1234, 1235, 1236, 1237, 1238, 1239, 1241, 1242, 1243, 1244, 1245, 1246, 1247, 1248, 1249, 1250, 1251, 1252, 1253, 1254, 1255, 1256, 1257, 1258, 1259, 1260, 1261, 1262, 1263, 1264, 1265, 1266, 1267, 1268, 1269, 1270, 1271, 1273, 1276, 1277, 1278, 1279, 1280, 1283, 1284, 1285, 1286, 1288, 1290, 1291, 1292, 1293, 1294, 1295, 1296, 1299, 1300, 1301, 1302, 1303, 1304, 1305, 1306, 1307, 1308, 1309, 1311, 1312, 1313, 1314, 1316, 1317, 1318, 1319, 1320, 1321, 1322, 1323, 1326, 1327, 1330, 1331, 1332, 1334, 1335, 1336, 1337, 1338, 1340, 1343, 1344, 1345, 1346, 1347, 1349, 1350, 1351, 1353, 1354, 1355, 1356, 1357, 1358, 1359, 1360, 1362, 1363, 1364, 1365, 1366, 1367, 1369, 1370, 1371, 1374, 1375, 1376, 1377, 1378, 1379, 1380, 1381, 1382, 1384, 1385, 1386, 1387, 1388, 1389, 1393, 1394, 1395, 1396, 1397, 1398, 1399, 1400, 1401, 1403, 1406, 1407, 1409, 1410, 1411, 1412, 1413, 1414, 1415, 1416, 1417, 1418, 1419, 1422, 1424, 1425, 1426, 1427, 1428, 1430, 1431, 1432, 1435, 1436, 1438, 1439, 1440, 1441, 1442, 1443, 1444, 1445, 1446, 1447, 1448, 1449, 1450, 1451, 1452, 1453, 1455, 1456, 1459, 1462, 1463, 1464, 1465, 1466, 1467, 1468, 1469, 1470, 1472, 1473, 1474, 1476, 1477, 1478, 1479, 1480, 1481, 1482, 1484, 1485, 1486, 1487, 1488, 1489, 1490, 1491, 1492, 1493, 1494, 1495, 1496, 1497, 1498, 1499, 1501, 1502, 1503, 1504, 1505, 1506, 1507, 1508, 1509, 1510, 1511, 1513, 1514, 1515, 1516, 1517, 1518, 1519, 1521, 1522, 1523, 1524, 1526, 1527, 1529, 1532, 1534, 1536, 1538, 1541, 1542, 1544, 1546, 1547, 1548, 1549, 1550, 1551, 1552, 1553, 1554, 1556, 1557, 1558, 1559, 1561, 1562, 1564, 1565, 1567, 1568, 1569, 1570, 1571, 1572, 1573, 1574, 1575, 1576, 1577, 1579, 1581, 1582, 1584, 1585, 1586, 1587, 1588, 1589, 1590, 1591, 1592, 1594, 1595, 1596, 1597, 1598, 1600, 1601, 1602, 1603, 1605, 1606, 1607, 1610, 1611, 1612, 1613, 1616, 1617, 1618, 1619, 1620, 1622, 1623, 1624, 1625, 1627, 1628, 1629, 1630, 1631, 1632, 1633, 1634, 1635, 1636, 1637, 1638, 1639, 1640, 1642, 1643, 1644, 1645, 1646, 1647, 1648, 1652, 1653, 1654, 1655, 1656, 1657, 1658, 1660, 1661, 1662, 1664, 1665, 1666, 1667, 1668, 1669, 1671, 1672, 1673, 1674, 1675, 1676, 1677, 1678, 1679, 1680, 1681, 1683, 1684, 1685, 1686, 1687, 1688, 1689, 1690, 1691, 1692, 1693, 1694, 1695, 1696, 1697, 1698, 1699, 1701, 1702, 1703, 1704, 1706, 1707, 1708, 1709, 1710, 1711, 1712, 1714, 1715, 1716, 1717, 1718, 1719, 1720, 1721, 1722, 1723, 1724, 1726, 1727, 1728, 1731, 1732, 1735, 1736, 1738, 1739, 1740, 1741, 1742, 1743, 1745, 1746, 1747, 1748, 1749, 1750, 1751, 1756, 1757, 1759, 1760, 1761, 1763, 1764, 1765, 1766, 1767, 1768, 1769, 1773, 1774, 1775, 1776, 1777, 1778, 1779, 1780, 1782, 1783, 1784, 1785, 1786, 1787, 1788, 1789, 1790, 1791, 1792, 1793, 1794, 1795, 1796, 1797, 1798, 1799, 1800, 1801, 1804, 1805, 1806, 1807, 1809, 1810, 1812, 1813, 1814, 1815, 1816, 1817, 1818, 1819, 1820, 1821, 1822, 1823, 1829, 1830, 1831, 1832, 1833, 1834, 1835, 1836, 1838, 1839, 1840, 1841, 1842, 1843, 1844, 1845, 1846, 1847, 1848, 1849, 1850, 1851, 1852, 1853, 1854, 1855, 1856, 1857, 1858, 1859, 1862, 1864, 1865, 1866, 1868, 1869, 1870, 1872, 1873, 1875, 1876, 1877, 1878, 1879, 1880, 1881, 1882, 1883, 1885, 1886, 1887, 1889, 1890, 1895, 1897, 1898, 1899, 1900, 1901, 1902, 1904, 1905, 1907, 1908, 1909, 1911, 1912, 1915, 1916, 1917, 1918, 1920, 1921, 1922, 1924, 1925, 1926, 1927, 1928, 1932, 1934, 1935, 1936, 1937, 1938, 1939, 1940, 1941, 1942, 1943, 1944, 1945, 1947, 1948, 1949, 1950, 1951, 1952, 1953, 1955, 1956, 1957, 1958, 1959, 1961, 1962, 1963, 1964, 1965, 1966, 1967, 1969, 1970, 1971, 1972, 1976, 1978, 1979, 1980, 1983, 1984, 1985, 1986, 1989, 1990, 1993, 1994, 1998, 1999, 2000, 2001, 2002, 2003, 2004, 2006, 2007, 2008, 2010, 2011, 2013, 2015, 2016, 2017, 2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026, 2027, 2028, 2029, 2030, 2032, 2033, 2034, 2035, 2036, 2037, 2038, 2039, 2040, 2042, 2043, 2044, 2045, 2047, 2048, 2050, 2051, 2052, 2053, 2054, 2055, 2056, 2057, 2058, 2059, 2061, 2062, 2063, 2064, 2066, 2067, 2068, 2069, 2070, 2071, 2072, 2073, 2074, 2075, 2076, 2077, 2080, 2081, 2082, 2083, 2084, 2086, 2088, 2089, 2090, 2092, 2093, 2095, 2096, 2097, 2098, 2099, 2100, 2101, 2103, 2104, 2106, 2108, 2109, 2110, 2111, 2112, 2113, 2114, 2115, 2116, 2117, 2118, 2119, 2121, 2122, 2123, 2124, 2125, 2126, 2127, 2128, 2129, 2130, 2134, 2135, 2136, 2137, 2138, 2139, 2140, 2141, 2142, 2143, 2145, 2146, 2147, 2148, 2151, 2153, 2154, 2155, 2156, 2157, 2158, 2159, 2160, 2163, 2164, 2165, 2168, 2169, 2170, 2171, 2174, 2175, 2176, 2178, 2179, 2181, 2182, 2183, 2184, 2185, 2186, 2187, 2188, 2190, 2191, 2193, 2194, 2195, 2196, 2197, 2199, 2200, 2201, 2202, 2203, 2204, 2205, 2207, 2208, 2209, 2211, 2212, 2213, 2215, 2217, 2218, 2219, 2220, 2221, 2223, 2224, 2225, 2226, 2227, 2228, 2230, 2231, 2232, 2234, 2235, 2236, 2237, 2238, 2239, 2240, 2242, 2243, 2244, 2245, 2246, 2247, 2248, 2249, 2250, 2251, 2252, 2253, 2254, 2255, 2256, 2257, 2258, 2259, 2260, 2261, 2262, 2264, 2265, 2266, 2267, 2269, 2271, 2272, 2274, 2275, 2276, 2277, 2278, 2279, 2280, 2281, 2282, 2283, 2284, 2285, 2287, 2289, 2290, 2291, 2292, 2293, 2294, 2295, 2296, 2297, 2298, 2299, 2301, 2302, 2303, 2304, 2305, 2306, 2307, 2309, 2310, 2311, 2312, 2314, 2316, 2317, 2318, 2319, 2320, 2321, 2322, 2323, 2325, 2326, 2327, 2329, 2330, 2331, 2332, 2333, 2334, 2335, 2336, 2337, 2338, 2339, 2340, 2341, 2342, 2343, 2345, 2347, 2348, 2350, 2351, 2354, 2355, 2357, 2358, 2359, 2360, 2361, 2362, 2365, 2367, 2369, 2370, 2371, 2372, 2373, 2374, 2375, 2376, 2377, 2378, 2379, 2380, 2381, 2382, 2383, 2386, 2387, 2388, 2389, 2391, 2392, 2393, 2394, 2395, 2396, 2397, 2398, 2399, 2400, 2401, 2402, 2403, 2404, 2405, 2406, 2407, 2408, 2409, 2410, 2411, 2412, 2413, 2414, 2416, 2417, 2418, 2419, 2420, 2421, 2422, 2423, 2424, 2425, 2426, 2427, 2428, 2429, 2431, 2432, 2433, 2434, 2435, 2437, 2438, 2439, 2440, 2442, 2443, 2444, 2446, 2447, 2448, 2452, 2453, 2454, 2456, 2457, 2459, 2460, 2462, 2463, 2464, 2465, 2466, 2469, 2471, 2472, 2473, 2474, 2476, 2477, 2478, 2479, 2480, 2483, 2486, 2488, 2489, 2490, 2492, 2493, 2494, 2495, 2496, 2497, 2499, 2500, 2501, 2502, 2503, 2504, 2505, 2506, 2507, 2508, 2509, 2511, 2512, 2513, 2516, 2518, 2520, 2521, 2523, 2524, 2525, 2526, 2528, 2529, 2530, 2531, 2532, 2533, 2534, 2535, 2537, 2539, 2540, 2541, 2542, 2543, 2544, 2545, 2547, 2549, 2550, 2551, 2552, 2553, 2554, 2555, 2556, 2557, 2558, 2560, 2562, 2563, 2566, 2567, 2568, 2569, 2570, 2571, 2572, 2573, 2574, 2575, 2576, 2577, 2578, 2579, 2580, 2582, 2584, 2585, 2588, 2589, 2591, 2592, 2593, 2594, 2595, 2596, 2597, 2598, 2599, 2600, 2601, 2602, 2603, 2604, 2605, 2606, 2607, 2608, 2609, 2611, 2612, 2613, 2614, 2615, 2616, 2617, 2618, 2619, 2620, 2621, 2622, 2624, 2626, 2627, 2630, 2631, 2632, 2633, 2634, 2635, 2636, 2637, 2638, 2639, 2640, 2641, 2642, 2643, 2644, 2645, 2646, 2647, 2648, 2649, 2650, 2651, 2653, 2654, 2655, 2656, 2657, 2660, 2661, 2663, 2664, 2665, 2666, 2667, 2670, 2672, 2673, 2674, 2675, 2676, 2677, 2678, 2679, 2680, 2681, 2682, 2683, 2684, 2685, 2686, 2687, 2688, 2689, 2690, 2692, 2693, 2694, 2695, 2696, 2698, 2699, 2701, 2702, 2703, 2705, 2707, 2708, 2709, 2711, 2712, 2713, 2714, 2715, 2716, 2717, 2718, 2719, 2720, 2722, 2723, 2724, 2726, 2727, 2728, 2729, 2732, 2733, 2734, 2735, 2737, 2738, 2739, 2740, 2742, 2744, 2745, 2746, 2747, 2748, 2750, 2751, 2752, 2753, 2754, 2755, 2756, 2758, 2759, 2760, 2761, 2762, 2763, 2764, 2765, 2766, 2767, 2768, 2769, 2770, 2773, 2774, 2775, 2776, 2778, 2779, 2780, 2781, 2783, 2785, 2787, 2788, 2789, 2790, 2791, 2792, 2793, 2794, 2795, 2796, 2799, 2800, 2801, 2802, 2803, 2804, 2805, 2806, 2807, 2808, 2809, 2811, 2813, 2818, 2819, 2820, 2821, 2822, 2823, 2824, 2825, 2826, 2827, 2828, 2829, 2831, 2832, 2833, 2834, 2835, 2837, 2838, 2839, 2840, 2841, 2843, 2844, 2845, 2846, 2847, 2848, 2849, 2850, 2851, 2853, 2854, 2855, 2856, 2857, 2858, 2859, 2860, 2861, 2862, 2863, 2867, 2868, 2869, 2870, 2871, 2872, 2873, 2874, 2875, 2876, 2878, 2879, 2880, 2881, 2882, 2883, 2884, 2885, 2887, 2888, 2889, 2890, 2891, 2892, 2893, 2894, 2895, 2896, 2898, 2899, 2900, 2902, 2903, 2904, 2905, 2906, 2908, 2910, 2911, 2912, 2913, 2914, 2915, 2917, 2918, 2919, 2920, 2921, 2922, 2923, 2924, 2926, 2927, 2929, 2930, 2932, 2933, 2934, 2935, 2938, 2939, 2940, 2941, 2942, 2944, 2945, 2946, 2947, 2948, 2949, 2950, 2951, 2952, 2953, 2954, 2955, 2956, 2957, 2958, 2959, 2960, 2961, 2962, 2963, 2964, 2965, 2966, 2968, 2969, 2970, 2971, 2973, 2974, 2975, 2978, 2979, 2980, 2981, 2982, 2983, 2984, 2985, 2986, 2987, 2988, 2989, 2990, 2991, 2992, 2994, 2995, 2996, 2997, 2998, 2999, 3000, 3001, 3002, 3004, 3005, 3006, 3008, 3009, 3010, 3012, 3014, 3015, 3016, 3017, 3018, 3019, 3021, 3022, 3023, 3024, 3025, 3026, 3027, 3028, 3029, 3030, 3031, 3032, 3033, 3034, 3035, 3036, 3037, 3038, 3039, 3040, 3041, 3042, 3043, 3044, 3046, 3047, 3048, 3050, 3051, 3052, 3053, 3055, 3056, 3057, 3058, 3059, 3060, 3061, 3062, 3063, 3064, 3065, 3066, 3067, 3068, 3069, 3070, 3072, 3074, 3075, 3077, 3078, 3079, 3080, 3081, 3084, 3085, 3087, 3088, 3089, 3090, 3091, 3092, 3093, 3094, 3095, 3096, 3097, 3098, 3099, 3101, 3102, 3103, 3104, 3105, 3106, 3107, 3108, 3109, 3110, 3111, 3112, 3114, 3116, 3117, 3118, 3119, 3122, 3123, 3124, 3125, 3126, 3127, 3129, 3130, 3131, 3132, 3133, 3134, 3135, 3136, 3137, 3138, 3139, 3140, 3141, 3142, 3143, 3144, 3145, 3146, 3148, 3149, 3150, 3151, 3152, 3153, 3155, 3156, 3157, 3158, 3159, 3160, 3161, 3162, 3163, 3164, 3165, 3166, 3167, 3168, 3169, 3171, 3172, 3173, 3174, 3175, 3176, 3177, 3178, 3179, 3180, 3181, 3182, 3183, 3184, 3185, 3186, 3187, 3188, 3191, 3192, 3193, 3195, 3196, 3197, 3198, 3200, 3201, 3203, 3204, 3205, 3206, 3207, 3209, 3210, 3211, 3212, 3214, 3215, 3216, 3217, 3218, 3219, 3220, 3222, 3223, 3224, 3225, 3226, 3227, 3229, 3231, 3232, 3233, 3234, 3235, 3236, 3237, 3238, 3240, 3241, 3243, 3245, 3246, 3247, 3248, 3249, 3251, 3252, 3253, 3254, 3255, 3256, 3257, 3258, 3259, 3260, 3261, 3262, 3263, 3264, 3265, 3267, 3268, 3269, 3270, 3271, 3272, 3273, 3274, 3276, 3277, 3278, 3279, 3282, 3283, 3284, 3285, 3286, 3287, 3288, 3290, 3291, 3292, 3293, 3294, 3296, 3297, 3299, 3300, 3301, 3302, 3303, 3304, 3306, 3308, 3309, 3310, 3312, 3313, 3315, 3316, 3317, 3318, 3319, 3320, 3321, 3322, 3324, 3325, 3326, 3327, 3329, 3330, 3331, 3332, 3334, 3335, 3336, 3337, 3339, 3340, 3341, 3342, 3343, 3344, 3345, 3346, 3347, 3348, 3350, 3351, 3352, 3353, 3354, 3355, 3356, 3357, 3358, 3360, 3362, 3365, 3366, 3367, 3369, 3370, 3371, 3372, 3373, 3375, 3376, 3377, 3378, 3379, 3380, 3381, 3382, 3384, 3385, 3386, 3387, 3388, 3389, 3390, 3391, 3392, 3393, 3394, 3395, 3396, 3399, 3400, 3401, 3402, 3403, 3404, 3405, 3406, 3407, 3409, 3410, 3412, 3413, 3414, 3416, 3417, 3418, 3419, 3420, 3421, 3422, 3423, 3424, 3425, 3426, 3427, 3428, 3429, 3430, 3431, 3432, 3435, 3436, 3437, 3438, 3439, 3440, 3442, 3443, 3444, 3445, 3446, 3449, 3450, 3453, 3454, 3456, 3457, 3460, 3461, 3462, 3463, 3464, 3465, 3466, 3467, 3468, 3469, 3471, 3472, 3473, 3474, 3475, 3476, 3477, 3478, 3479, 3480, 3481, 3482, 3483, 3484, 3485, 3486, 3487, 3488, 3491, 3492, 3493, 3494, 3495, 3496, 3497, 3498, 3499, 3500, 3501, 3502, 3503, 3504, 3505, 3506, 3507, 3508, 3509, 3510, 3511, 3513, 3514, 3515, 3516, 3517, 3519, 3520, 3521, 3522, 3523, 3524, 3525, 3526, 3527, 3528, 3529, 3530, 3531, 3532, 3533, 3534, 3535, 3536, 3537, 3538, 3539, 3542, 3543, 3544, 3545, 3546, 3550, 3551, 3552, 3553, 3554, 3555, 3557, 3558, 3559, 3561, 3563, 3564, 3565, 3566, 3567, 3568, 3570, 3571, 3572, 3573, 3574, 3575, 3576, 3577, 3578, 3579, 3581, 3584, 3585, 3586, 3587, 3588, 3590, 3591, 3592, 3593, 3595, 3596, 3597, 3598, 3600, 3601, 3602, 3603, 3604, 3605, 3606, 3607, 3608, 3609, 3610, 3611, 3612, 3613, 3614, 3615, 3617, 3618, 3619, 3620, 3621, 3624, 3625, 3626, 3627, 3629, 3630, 3631, 3633, 3634, 3635, 3636, 3637, 3638, 3639, 3640, 3641, 3642, 3643, 3644, 3645, 3646, 3647, 3648, 3649, 3650, 3651, 3652, 3653, 3654, 3655, 3656, 3657, 3659, 3660, 3661, 3662, 3664, 3666, 3667, 3668, 3669, 3670, 3671, 3672, 3674, 3675, 3677, 3678, 3679, 3681, 3683, 3684, 3685, 3686, 3687, 3689, 3690, 3691, 3693, 3694, 3695, 3697, 3700, 3701, 3702, 3703, 3704, 3705, 3706, 3707, 3708, 3709, 3710, 3711, 3712, 3713, 3714, 3715, 3716, 3717, 3719, 3720, 3721, 3723, 3725, 3726, 3727, 3728, 3729, 3730, 3731, 3733, 3734, 3735, 3736, 3737, 3739, 3740, 3741, 3743, 3745, 3747, 3748, 3749, 3750, 3751, 3752, 3753, 3754, 3755, 3756, 3758, 3759, 3760, 3761, 3762, 3763, 3764, 3765, 3766, 3769, 3770, 3771, 3772, 3773, 3774, 3775, 3776, 3777, 3778, 3779, 3780, 3782, 3783, 3784, 3785, 3786, 3787, 3788, 3789, 3790, 3791, 3793, 3795, 3797, 3798, 3799, 3800, 3801, 3803, 3804, 3805, 3806, 3808, 3810, 3811, 3812, 3813, 3814, 3815, 3816, 3817, 3818, 3819, 3820, 3822, 3823, 3824, 3825, 3826, 3827, 3828, 3829, 3830, 3831, 3833, 3834, 3835, 3836, 3837, 3838, 3839, 3840, 3841, 3842, 3843, 3844, 3845, 3846, 3849, 3850, 3851, 3852, 3853, 3854, 3856, 3857, 3858, 3859, 3860, 3861, 3862, 3863, 3864, 3865, 3866, 3868, 3869, 3870, 3871, 3872, 3873, 3874, 3876, 3877, 3878, 3879, 3880, 3881, 3882, 3883, 3884, 3885, 3886, 3887, 3889, 3891, 3892, 3893, 3894, 3895, 3896, 3897, 3898, 3900, 3901, 3902, 3903, 3904, 3905, 3906, 3907, 3908, 3909, 3913, 3914, 3915, 3917, 3918, 3919, 3921, 3922, 3923, 3924, 3925, 3926, 3927, 3928, 3929, 3931, 3932, 3933, 3934, 3935, 3936, 3938, 3940, 3941, 3942, 3944, 3945, 3946, 3948, 3949, 3950, 3951, 3954, 3955, 3956, 3957, 3958, 3959, 3960, 3961, 3962, 3963, 3964, 3965, 3966, 3967, 3968, 3969, 3970, 3971, 3972, 3974, 3975, 3976, 3978, 3980, 3982, 3983, 3984, 3986, 3987, 3988, 3989, 3990, 3991, 3993, 3994, 3995, 3997, 3998, 3999, 4000, 4003, 4005, 4006, 4007, 4008, 4009, 4010, 4011, 4013, 4014, 4016, 4017, 4019, 4020, 4021, 4022, 4023, 4024, 4025, 4026, 4027, 4028, 4029, 4031, 4032, 4034, 4036, 4037, 4038, 4040, 4041, 4042, 4043, 4045, 4046, 4047, 4048, 4049, 4051, 4052, 4053, 4054, 4055, 4056, 4057, 4058, 4060, 4061, 4062, 4065, 4067, 4069, 4070, 4072, 4073, 4075, 4076, 4077, 4078, 4079, 4080, 4081, 4083, 4084, 4085, 4086, 4087, 4088, 4089, 4090, 4091, 4092, 4093, 4094, 4095, 4096, 4097, 4098, 4099, 4100, 4101, 4102, 4103, 4104, 4105, 4107, 4109, 4110, 4111, 4112, 4113, 4114, 4115, 4118, 4119, 4120, 4121, 4123, 4124, 4126, 4127, 4129, 4131, 4133, 4134, 4135, 4136, 4137, 4138, 4139, 4140, 4142, 4143, 4144, 4145, 4146, 4147, 4148, 4151, 4152, 4153, 4154, 4155, 4156, 4157, 4158, 4159, 4160, 4161, 4162, 4164, 4166, 4168, 4169, 4170, 4171, 4172, 4173, 4174, 4175, 4176, 4177, 4178, 4179, 4180, 4181, 4182, 4183, 4184, 4185, 4187, 4188, 4189, 4197, 4198, 4199, 4200, 4201, 4203, 4205, 4206, 4207, 4209, 4211, 4212, 4215, 4216, 4217, 4218, 4219, 4220, 4221, 4224, 4226, 4227, 4228, 4231, 4232, 4233, 4234, 4235, 4236, 4238, 4239, 4240, 4242, 4243, 4244, 4245, 4247, 4248, 4249, 4250, 4251, 4252, 4253, 4255, 4256, 4259, 4260, 4261, 4262, 4263, 4265, 4266, 4268, 4269, 4270, 4271, 4272, 4273, 4275, 4276, 4277, 4278, 4279, 4280, 4281, 4282, 4283, 4284, 4285, 4286, 4289, 4290, 4292, 4294, 4295, 4296, 4298, 4299, 4300, 4301, 4302, 4303, 4307, 4308, 4309, 4311, 4312, 4313, 4314, 4316, 4317, 4318, 4319, 4321, 4322, 4323, 4324, 4325, 4326, 4327, 4328, 4329, 4330, 4331, 4333, 4334, 4335, 4336, 4337, 4340, 4342, 4343, 4344, 4345, 4346, 4347, 4348, 4349, 4350, 4351, 4352, 4353, 4355, 4356, 4357, 4358, 4359, 4361, 4362, 4363, 4364, 4365, 4368, 4369, 4370, 4371, 4373, 4374, 4375, 4377, 4378, 4381, 4382, 4383, 4385, 4386, 4387, 4388, 4389, 4390, 4393, 4394, 4396, 4397, 4399, 4400, 4403, 4406, 4407, 4410, 4411, 4412, 4413, 4414, 4416, 4417, 4418, 4420, 4421, 4422, 4423, 4424, 4425, 4426, 4427, 4428, 4429, 4431, 4432, 4433, 4434, 4435, 4436, 4437, 4438, 4440, 4441, 4442, 4443, 4444, 4446, 4447, 4448, 4453, 4455, 4457, 4458, 4459, 4460, 4463, 4464, 4465, 4466, 4467, 4469, 4470, 4472, 4473, 4474, 4475, 4476, 4477, 4479, 4480, 4482, 4483, 4484, 4486, 4487, 4489, 4490, 4492, 4493, 4494, 4495, 4496, 4498, 4499, 4500, 4501, 4502, 4503, 4504, 4505, 4506, 4507, 4508, 4511, 4512, 4513, 4514, 4515, 4516, 4517, 4519, 4520, 4521, 4523, 4524, 4525, 4526, 4527, 4529, 4530, 4532, 4533, 4534, 4535, 4536, 4537, 4538, 4541, 4542, 4543, 4544, 4545, 4546, 4547, 4548, 4549, 4550, 4551, 4552, 4554, 4556, 4557, 4558, 4559, 4560, 4561, 4562, 4563, 4564, 4565, 4566, 4567, 4568, 4570, 4571, 4572, 4573, 4575, 4576, 4577, 4578, 4579, 4581, 4582, 4583, 4585, 4586, 4587, 4588, 4589, 4590, 4591, 4592, 4593, 4594, 4597, 4598, 4599, 4601, 4604, 4606, 4607, 4608, 4610, 4611, 4612, 4613, 4615, 4616, 4617, 4618, 4619, 4620, 4621, 4622, 4623, 4624, 4626, 4627, 4628, 4629, 4631, 4632, 4633, 4634, 4636, 4637, 4641, 4642, 4643, 4644, 4645, 4646, 4650, 4651, 4652, 4653, 4654, 4655, 4656, 4658, 4660, 4661, 4662, 4663, 4664, 4666, 4667, 4669, 4671, 4672, 4674, 4676, 4677, 4678, 4679, 4680, 4681, 4684, 4685, 4686, 4687, 4688, 4689, 4690, 4691, 4692, 4693, 4694, 4695, 4696, 4697, 4698, 4699, 4700, 4702, 4703, 4704, 4706, 4707, 4709, 4710, 4711, 4712, 4713, 4714, 4715, 4716, 4718, 4724, 4725, 4726, 4727, 4728, 4730, 4731, 4732, 4733, 4734, 4735, 4737, 4739, 4740, 4741, 4743, 4745, 4746, 4747, 4750, 4751, 4752, 4756, 4757, 4758, 4761, 4762, 4763, 4765, 4766, 4767, 4768, 4769, 4770, 4772, 4773, 4774, 4775, 4776, 4777, 4778, 4779, 4781, 4782, 4783, 4784, 4785, 4787, 4788, 4791, 4795, 4796, 4797, 4798, 4799, 4803, 4804, 4805, 4807, 4810, 4811, 4813, 4816, 4817, 4818, 4819, 4820, 4821, 4823, 4825, 4828, 4829, 4830, 4832, 4833, 4834, 4835, 4837, 4838, 4839, 4841, 4842, 4844, 4845, 4846, 4850, 4851, 4852, 4853, 4856, 4857, 4858, 4860, 4861, 4862, 4863, 4865, 4867, 4869, 4870, 4871, 4872, 4873, 4875, 4876, 4877, 4879, 4880, 4881, 4882, 4884, 4885, 4886, 4887, 4888, 4890, 4891, 4892, 4893, 4894, 4895, 4896, 4897, 4898, 4899, 4900, 4901, 4902, 4903, 4904, 4905, 4906, 4908, 4909, 4910, 4911, 4912, 4915, 4916, 4919, 4920, 4921, 4922, 4923, 4924, 4927, 4928, 4929, 4930, 4931, 4932, 4933, 4934, 4935, 4936, 4937, 4938, 4939, 4940, 4941, 4942, 4943, 4945, 4946, 4947, 4949, 4950, 4951, 4952, 4955, 4956, 4958, 4959, 4960, 4961, 4962, 4963, 4964, 4965, 4966, 4967, 4969, 4970, 4972, 4973, 4974, 4975, 4976, 4977, 4978, 4979, 4980, 4981, 4982, 4983, 4984, 4986, 4987, 4988, 4989, 4990, 4991, 4992, 4993, 4994, 4995, 4996, 4997, 4998, 4999, 5000, 5001, 5002, 5003, 5004, 5005, 5006, 5007, 5008, 5010, 5011, 5012, 5014, 5016, 5017, 5018, 5019, 5020, 5021, 5022, 5023, 5024, 5025, 5026, 5027, 5028, 5029, 5030, 5032, 5033, 5034, 5035, 5037, 5038, 5039, 5040, 5041, 5042, 5043, 5044, 5046, 5047, 5048, 5050, 5051, 5052, 5053, 5054, 5055, 5056, 5057, 5059, 5060, 5061, 5062, 5063, 5064, 5065, 5066, 5067, 5068, 5069, 5070, 5071, 5072, 5073, 5074, 5075, 5076, 5078, 5079, 5081, 5083, 5084, 5085, 5087, 5088, 5089, 5091, 5092, 5093, 5094, 5095, 5096, 5097, 5098, 5099, 5100, 5101, 5102, 5103, 5105, 5106, 5107, 5110, 5111, 5112, 5113, 5114, 5115, 5117, 5118, 5119, 5120, 5122, 5123, 5124, 5125, 5127, 5128, 5129, 5131, 5132, 5133, 5134, 5135, 5136, 5137, 5138, 5139, 5140, 5141, 5142, 5143, 5144, 5146, 5147, 5149, 5150, 5152, 5153, 5154, 5156, 5157, 5159, 5160, 5161, 5162, 5163, 5165, 5166, 5168, 5172, 5173, 5174, 5175, 5176, 5177, 5178, 5183, 5184, 5186, 5189, 5190, 5191, 5192, 5193, 5194, 5195, 5196, 5197, 5198, 5199, 5201, 5202, 5203, 5204, 5205, 5206, 5207, 5208, 5210, 5211, 5212, 5213, 5214, 5215, 5216, 5218, 5219, 5221, 5222, 5225, 5226, 5227, 5228, 5229, 5230, 5231, 5232, 5233, 5234, 5235, 5236, 5237, 5238, 5239, 5240, 5241, 5242, 5243, 5244, 5246, 5247, 5248, 5249, 5251, 5252, 5253, 5255, 5256, 5257, 5258, 5259, 5260, 5262, 5263, 5264, 5265, 5267, 5272, 5273, 5274, 5275, 5276, 5277, 5278, 5280, 5282, 5283, 5284, 5286, 5287, 5288, 5291, 5292, 5293, 5294, 5295, 5296, 5297, 5298, 5299, 5300, 5301, 5302, 5303, 5304, 5305, 5306, 5307, 5309, 5310, 5311, 5312, 5313, 5314, 5318, 5320, 5321, 5322, 5323, 5324, 5325, 5326, 5327, 5328, 5329, 5330, 5332, 5333, 5334, 5335, 5336, 5337, 5338, 5339, 5341, 5342, 5343, 5345, 5346, 5347, 5348, 5349, 5350, 5351, 5352, 5353, 5354, 5355, 5356, 5357, 5358, 5359, 5360, 5362, 5363, 5364, 5365, 5366, 5367, 5368, 5369, 5370, 5372, 5373, 5375, 5376, 5377, 5379, 5380, 5382, 5385, 5386, 5387, 5389, 5390, 5391, 5392, 5393, 5395, 5396, 5398, 5400, 5401, 5402, 5404, 5405, 5406, 5407, 5408, 5409, 5410, 5411, 5412, 5413, 5414, 5415, 5416, 5417, 5418, 5419, 5421, 5423, 5424, 5425, 5426, 5427, 5428, 5430, 5431, 5432, 5433, 5434, 5435, 5436, 5437, 5438, 5439, 5440, 5442, 5443, 5444, 5446, 5448, 5449, 5450, 5451, 5453, 5454, 5455, 5457, 5458, 5460, 5462, 5463, 5465, 5466, 5468, 5469, 5470, 5471, 5472, 5473, 5474, 5475, 5476, 5477, 5479, 5480, 5481, 5482, 5484, 5485, 5487, 5489, 5490, 5491, 5492, 5493, 5494, 5496, 5497, 5498, 5500, 5501, 5502, 5503, 5504, 5507, 5508, 5510, 5511, 5512, 5513, 5514, 5515, 5516, 5520, 5522, 5523, 5527, 5528, 5529, 5530, 5531, 5533, 5535, 5537, 5538, 5539, 5543, 5544, 5546, 5548, 5549, 5550, 5551, 5552, 5553, 5554, 5558, 5559, 5561, 5562, 5563, 5565, 5566, 5567, 5568, 5569, 5570, 5571, 5573, 5574, 5575, 5576, 5578, 5579, 5580, 5581, 5582, 5583, 5584, 5585, 5586, 5587, 5588, 5589, 5590, 5591, 5592, 5594, 5595, 5597, 5598, 5599, 5600, 5602, 5603, 5604, 5606, 5607, 5608, 5609, 5611, 5612, 5613, 5614, 5615, 5616, 5617, 5619, 5620, 5621, 5623, 5624, 5625, 5626, 5627, 5628, 5630, 5631, 5632, 5633, 5634, 5635, 5636, 5638, 5640, 5641, 5642, 5643, 5644, 5645, 5646, 5647, 5648, 5649, 5650, 5651, 5652, 5654, 5655, 5656, 5658, 5659, 5660, 5661, 5662, 5663, 5664, 5665, 5666, 5667, 5668, 5669, 5670, 5671, 5672, 5673, 5674, 5675, 5676, 5677, 5678, 5679, 5680, 5681, 5685, 5686, 5687, 5688, 5689, 5690, 5691, 5692, 5694, 5695, 5696, 5697, 5698, 5699, 5701, 5702, 5703, 5704, 5705, 5706, 5707, 5708, 5709, 5710, 5711, 5712, 5713, 5714, 5716, 5717, 5718, 5719, 5720, 5721, 5722, 5724, 5725, 5726, 5727, 5728, 5731, 5732, 5734, 5735, 5736, 5737, 5738, 5739, 5740, 5741, 5742, 5743, 5744, 5745, 5746, 5748, 5749, 5750, 5751, 5752, 5753, 5754, 5755, 5756, 5757, 5759, 5760, 5761, 5762, 5763, 5764, 5765, 5766, 5767, 5768, 5769, 5770, 5771, 5772, 5773, 5774, 5777, 5778, 5779, 5780, 5781, 5782, 5783, 5784, 5785, 5786, 5787, 5788, 5791, 5792, 5793, 5794, 5795, 5796, 5797, 5799, 5800, 5801, 5802, 5803, 5804, 5805, 5806, 5807, 5808, 5810, 5812, 5813, 5814, 5815, 5816, 5817, 5819, 5822, 5823, 5824, 5825, 5826, 5827, 5829, 5830, 5831, 5834, 5835, 5836, 5837, 5838, 5839, 5840, 5841, 5842, 5843, 5846, 5847, 5848, 5849, 5850, 5851, 5852, 5853, 5854, 5856, 5858, 5859, 5860, 5863, 5864, 5865, 5866, 5868, 5869, 5870, 5871, 5872, 5873, 5876, 5877, 5878, 5879, 5881, 5882, 5883, 5884, 5886, 5887, 5889, 5891, 5892, 5893, 5894, 5896, 5897, 5898, 5899, 5900, 5901, 5902, 5905, 5907, 5908, 5910, 5912, 5913, 5914, 5916, 5917, 5918, 5919, 5920, 5921, 5922, 5923, 5924, 5925, 5926, 5929, 5930, 5932, 5933, 5934, 5935, 5937, 5938, 5939, 5940, 5944, 5945, 5946, 5947, 5948, 5949, 5951, 5952, 5953, 5954, 5955, 5957, 5958, 5959, 5960, 5961, 5963, 5964, 5965, 5966, 5967, 5968, 5969, 5971, 5973, 5974, 5975, 5976, 5977, 5978, 5979, 5980, 5984, 5985, 5986, 5987, 5988, 5989, 5990, 5991, 5993, 5994, 5995, 5996, 5997, 5998, 5999, 6000, 6002, 6003, 6004, 6005, 6006, 6007, 6009, 6010, 6011, 6012, 6013, 6014, 6015, 6016, 6018, 6019, 6020, 6022, 6023, 6024, 6026, 6027, 6028, 6029, 6030, 6032, 6033, 6035, 6036, 6038, 6039, 6041, 6042, 6043, 6044, 6045, 6046, 6047, 6048, 6049, 6051, 6053, 6054, 6055, 6056, 6057, 6058, 6060, 6061, 6062, 6063, 6064, 6066, 6068, 6070, 6071, 6073, 6074, 6075, 6076, 6077, 6078, 6079, 6081, 6082, 6083, 6085, 6086, 6087, 6088, 6089, 6090, 6091, 6092, 6093, 6094, 6095, 6096, 6097, 6098, 6099, 6102, 6103, 6104, 6105, 6106, 6107, 6108, 6109, 6110, 6111, 6112, 6115, 6116, 6117, 6118, 6119, 6120, 6121, 6122, 6123, 6124, 6125, 6126, 6127, 6128, 6130, 6132, 6133, 6134, 6135, 6136, 6137, 6139, 6140, 6141, 6142, 6144, 6145, 6147, 6148, 6150, 6152, 6153, 6155, 6156, 6158, 6159, 6161, 6163, 6164, 6165, 6166, 6167, 6170, 6172, 6173, 6174, 6175, 6176, 6177, 6178, 6180, 6181, 6183, 6184, 6185, 6186, 6187, 6188, 6190, 6191, 6192, 6193, 6194, 6195, 6197, 6198, 6199, 6201, 6202, 6203, 6205, 6206, 6207, 6208, 6209, 6210, 6213, 6214, 6215, 6218, 6219, 6220, 6221, 6222, 6224, 6225, 6226, 6227, 6228, 6229, 6230, 6231, 6232, 6233, 6234, 6235, 6236, 6237, 6238, 6239, 6240, 6242, 6243, 6244, 6245, 6246, 6247, 6248, 6249, 6251, 6252, 6253, 6254, 6255, 6256, 6257, 6258, 6261, 6262, 6263, 6264, 6266, 6268, 6270, 6271, 6272, 6273, 6274, 6275, 6276, 6277, 6278, 6279, 6280, 6281, 6282, 6285, 6286, 6289, 6291, 6292, 6293, 6294, 6295, 6296, 6297, 6298, 6299, 6300, 6301, 6303, 6304, 6305, 6306, 6307, 6309, 6310, 6311, 6312, 6313, 6314, 6316, 6317, 6318, 6319, 6320, 6321, 6322, 6324, 6325, 6326, 6327, 6328, 6329, 6330, 6331, 6332, 6333, 6334, 6335, 6336, 6337, 6338, 6339, 6341, 6342, 6343, 6344, 6345, 6346, 6348, 6349, 6350, 6351, 6353, 6354, 6355, 6356, 6357, 6358, 6360, 6361, 6362, 6363, 6364, 6365, 6366, 6367, 6368, 6369, 6370, 6372, 6373, 6376, 6379, 6380, 6381, 6382, 6383, 6384, 6385, 6386, 6387, 6388, 6389, 6391, 6392, 6393, 6394, 6395, 6396, 6397, 6399, 6400, 6401, 6402, 6403, 6405, 6406, 6408, 6409, 6410, 6411, 6412, 6414, 6415, 6417, 6419, 6420, 6421, 6423, 6424, 6425, 6428, 6429, 6430, 6431, 6432, 6433, 6434, 6435, 6436, 6437, 6439, 6440, 6441, 6442, 6443, 6444, 6446, 6447, 6448, 6449, 6450, 6451, 6452, 6453, 6454, 6455, 6456, 6457, 6458, 6459, 6460, 6462, 6463, 6465, 6466, 6467, 6468, 6469, 6470, 6471, 6472, 6473, 6475, 6478, 6479, 6480, 6481, 6482, 6483, 6484, 6485, 6486, 6491, 6492, 6493, 6494, 6496, 6497, 6498, 6499, 6500, 6501, 6502, 6504, 6505, 6506, 6507, 6508, 6510, 6511, 6512, 6513, 6514, 6515, 6516, 6517, 6518, 6519, 6520, 6521, 6523, 6524, 6525, 6526, 6527, 6529, 6530, 6532, 6533, 6534, 6535, 6536, 6538, 6539, 6540, 6541, 6542, 6543, 6546, 6550, 6552, 6553, 6554, 6555, 6557, 6558, 6559, 6560, 6561, 6562, 6563, 6564, 6565, 6566, 6568, 6569, 6572, 6573, 6574, 6575, 6577, 6578, 6579, 6580, 6583, 6585, 6587, 6589, 6591, 6592, 6593, 6595, 6596, 6597, 6598, 6599, 6600, 6601, 6602, 6604, 6605, 6606, 6608, 6609, 6610, 6612, 6614, 6615, 6616, 6617, 6619, 6620, 6621, 6622, 6623, 6624, 6625, 6626, 6627, 6628, 6629, 6630, 6632, 6634, 6636, 6638, 6639, 6640, 6641, 6642, 6644, 6646, 6648, 6649, 6650, 6651, 6653, 6654, 6655, 6656, 6657, 6658, 6659, 6660, 6661, 6662, 6665, 6666, 6668, 6669, 6670, 6676, 6677, 6678, 6680, 6681, 6682, 6683, 6685, 6686, 6687, 6688, 6689, 6691, 6692, 6694, 6695, 6696, 6697, 6698, 6699, 6700, 6701, 6704, 6705, 6706, 6708, 6709, 6710, 6711, 6712, 6713, 6714]
DEV_NUMS = []
HOLDOUT_NUMS = [37, 57, 61, 65, 72, 77, 78, 84, 91, 93, 106, 108, 110, 111, 113, 117, 119, 124, 127, 130, 132, 134, 143, 145, 155, 156, 166, 169, 174, 177, 185, 192, 193, 203, 205, 208, 209, 213, 214, 221, 224, 225, 226, 230, 231, 234, 238, 239, 241, 245, 247, 251, 252, 260, 266, 267, 271, 273, 277, 280, 281, 290, 299, 300, 303, 304, 307, 309, 312, 316, 317, 319, 322, 323, 325, 335, 339, 346, 347, 355, 356, 360, 363, 365, 370, 371, 381, 387, 388, 390, 401, 428, 432, 442, 463, 475, 480, 481, 484, 500, 505, 508, 515, 517, 519, 524, 528, 529, 531, 537, 542, 551, 558, 567, 569, 570, 573, 574, 575, 584, 594, 599, 601, 603, 613, 614, 625, 641, 649, 650, 652, 670, 673, 675, 698, 703, 718, 723, 742, 746, 756, 758, 765, 775, 776, 778, 785, 786, 787, 792, 797, 803, 807, 827, 829, 833, 842, 846, 849, 858, 863, 870, 871, 873, 875, 882, 893, 909, 912, 913, 937, 939, 940, 944, 949, 968, 979, 983, 985, 1000, 1003, 1005, 1006, 1012, 1018, 1022, 1033, 1034, 1039, 1042, 1045, 1048, 1049, 1052, 1061, 1065, 1070, 1072, 1075, 1079, 1080, 1083, 1087, 1092, 1094, 1097, 1106, 1116, 1117, 1118, 1119, 1120, 1123, 1127, 1134, 1139, 1145, 1149, 1150, 1152, 1155, 1156, 1166, 1171, 1172, 1174, 1180, 1181, 1184, 1193, 1197, 1199, 1200, 1201, 1211, 1213, 1218, 1221, 1223, 1224, 1230, 1240, 1272, 1274, 1275, 1281, 1282, 1287, 1289, 1297, 1298, 1315, 1324, 1325, 1328, 1329, 1333, 1339, 1341, 1342, 1348, 1352, 1361, 1368, 1372, 1373, 1383, 1390, 1391, 1392, 1402, 1404, 1405, 1408, 1420, 1421, 1423, 1429, 1433, 1434, 1437, 1454, 1457, 1458, 1460, 1461, 1471, 1475, 1483, 1500, 1512, 1520, 1525, 1528, 1530, 1531, 1533, 1535, 1537, 1539, 1540, 1543, 1545, 1555, 1560, 1563, 1566, 1578, 1580, 1583, 1593, 1599, 1604, 1608, 1609, 1614, 1615, 1621, 1626, 1641, 1649, 1650, 1651, 1659, 1663, 1670, 1682, 1700, 1705, 1713, 1725, 1729, 1730, 1733, 1734, 1737, 1744, 1752, 1753, 1754, 1755, 1758, 1762, 1770, 1771, 1772, 1781, 1802, 1803, 1811, 1824, 1825, 1826, 1827, 1828, 1837, 1860, 1861, 1863, 1867, 1871, 1874, 1884, 1888, 1891, 1892, 1893, 1894, 1896, 1903, 1906, 1910, 1913, 1914, 1919, 1923, 1929, 1930, 1931, 1933, 1946, 1954, 1960, 1968, 1973, 1974, 1975, 1977, 1981, 1982, 1987, 1988, 1991, 1992, 1995, 1996, 1997, 2005, 2009, 2012, 2014, 2018, 2031, 2041, 2046, 2049, 2060, 2065, 2078, 2079, 2085, 2087, 2091, 2094, 2102, 2105, 2107, 2120, 2131, 2132, 2133, 2144, 2149, 2150, 2152, 2161, 2162, 2166, 2167, 2172, 2173, 2177, 2180, 2189, 2192, 2198, 2206, 2210, 2214, 2216, 2222, 2229, 2233, 2241, 2263, 2268, 2270, 2273, 2286, 2288, 2300, 2308, 2313, 2315, 2324, 2328, 2344, 2346, 2349, 2352, 2353, 2356, 2363, 2364, 2366, 2368, 2384, 2385, 2390, 2415, 2430, 2436, 2441, 2445, 2449, 2450, 2451, 2455, 2458, 2461, 2467, 2468, 2470, 2475, 2481, 2482, 2484, 2485, 2487, 2491, 2498, 2510, 2514, 2515, 2517, 2519, 2522, 2527, 2536, 2538, 2546, 2548, 2559, 2561, 2564, 2565, 2581, 2583, 2586, 2587, 2590, 2610, 2623, 2625, 2628, 2629, 2652, 2658, 2659, 2662, 2668, 2669, 2671, 2691, 2697, 2700, 2704, 2706, 2710, 2721, 2725, 2730, 2731, 2736, 2741, 2743, 2749, 2757, 2771, 2772, 2777, 2782, 2784, 2786, 2797, 2798, 2810, 2812, 2814, 2815, 2816, 2817, 2830, 2836, 2842, 2852, 2864, 2865, 2866, 2877, 2886, 2897, 2901, 2907, 2909, 2916, 2925, 2928, 2931, 2936, 2937, 2943, 2967, 2972, 2976, 2977, 2993, 3003, 3007, 3011, 3013, 3020, 3045, 3049, 3054, 3071, 3073, 3076, 3082, 3083, 3086, 3100, 3113, 3115, 3120, 3121, 3128, 3147, 3154, 3170, 3189, 3190, 3194, 3199, 3202, 3208, 3213, 3221, 3228, 3230, 3239, 3242, 3244, 3250, 3266, 3275, 3280, 3281, 3289, 3295, 3298, 3305, 3311, 3314, 3323, 3328, 3333, 3338, 3349, 3359, 3361, 3363, 3364, 3368, 3374, 3383, 3397, 3398, 3408, 3411, 3415, 3433, 3441, 3447, 3448, 3451, 3452, 3455, 3458, 3459, 3470, 3489, 3490, 3512, 3518, 3540, 3541, 3547, 3548, 3549, 3556, 3560, 3562, 3569, 3580, 3582, 3583, 3589, 3594, 3599, 3616, 3622, 3623, 3628, 3632, 3658, 3663, 3665, 3673, 3676, 3680, 3682, 3688, 3692, 3696, 3698, 3699, 3718, 3724, 3732, 3738, 3742, 3744, 3746, 3757, 3767, 3768, 3781, 3792, 3794, 3796, 3802, 3807, 3809, 3821, 3832, 3847, 3848, 3855, 3867, 3875, 3888, 3890, 3899, 3910, 3911, 3912, 3916, 3920, 3930, 3937, 3939, 3943, 3947, 3952, 3953, 3973, 3977, 3979, 3981, 3985, 3992, 3996, 4001, 4002, 4004, 4015, 4018, 4030, 4033, 4035, 4039, 4044, 4050, 4059, 4063, 4064, 4066, 4068, 4071, 4074, 4082, 4106, 4108, 4116, 4117, 4122, 4125, 4128, 4130, 4132, 4141, 4149, 4150, 4163, 4165, 4167, 4186, 4190, 4191, 4192, 4193, 4194, 4195, 4196, 4202, 4204, 4208, 4210, 4213, 4214, 4222, 4223, 4225, 4229, 4230, 4237, 4241, 4246, 4254, 4257, 4258, 4264, 4267, 4274, 4287, 4288, 4291, 4293, 4297, 4304, 4305, 4306, 4310, 4315, 4320, 4332, 4338, 4339, 4341, 4354, 4360, 4366, 4367, 4372, 4376, 4379, 4380, 4384, 4391, 4392, 4395, 4398, 4401, 4402, 4404, 4405, 4408, 4409, 4415, 4419, 4430, 4439, 4445, 4449, 4450, 4451, 4452, 4454, 4456, 4461, 4462, 4468, 4471, 4478, 4481, 4485, 4488, 4491, 4497, 4509, 4510, 4518, 4522, 4528, 4531, 4539, 4540, 4553, 4555, 4569, 4574, 4580, 4584, 4595, 4596, 4600, 4602, 4603, 4605, 4609, 4614, 4625, 4630, 4635, 4638, 4639, 4640, 4647, 4648, 4649, 4657, 4659, 4665, 4668, 4670, 4673, 4675, 4682, 4683, 4701, 4705, 4708, 4717, 4719, 4720, 4721, 4722, 4723, 4729, 4736, 4738, 4742, 4744, 4748, 4749, 4753, 4754, 4755, 4759, 4760, 4764, 4771, 4780, 4786, 4789, 4790, 4792, 4793, 4794, 4800, 4801, 4802, 4806, 4808, 4809, 4812, 4814, 4815, 4822, 4824, 4826, 4827, 4831, 4836, 4840, 4843, 4847, 4848, 4849, 4854, 4855, 4859, 4864, 4866, 4868, 4874, 4878, 4883, 4889, 4907, 4913, 4914, 4917, 4918, 4925, 4926, 4944, 4948, 4953, 4954, 4957, 4968, 4971, 4985, 5009, 5013, 5015, 5031, 5036, 5045, 5049, 5058, 5077, 5080, 5082, 5086, 5090, 5104, 5108, 5109, 5116, 5121, 5126, 5130, 5145, 5148, 5151, 5155, 5158, 5164, 5167, 5169, 5170, 5171, 5179, 5180, 5181, 5182, 5185, 5187, 5188, 5200, 5209, 5217, 5220, 5223, 5224, 5245, 5250, 5254, 5261, 5266, 5268, 5269, 5270, 5271, 5279, 5281, 5285, 5289, 5290, 5308, 5315, 5316, 5317, 5319, 5331, 5340, 5344, 5361, 5371, 5374, 5378, 5381, 5383, 5384, 5388, 5394, 5397, 5399, 5403, 5420, 5422, 5429, 5441, 5445, 5447, 5452, 5456, 5459, 5461, 5464, 5467, 5478, 5483, 5486, 5488, 5495, 5499, 5505, 5506, 5509, 5517, 5518, 5519, 5521, 5524, 5525, 5526, 5532, 5534, 5536, 5540, 5541, 5542, 5545, 5547, 5555, 5556, 5557, 5560, 5564, 5572, 5577, 5593, 5596, 5601, 5605, 5610, 5618, 5622, 5629, 5637, 5639, 5653, 5657, 5682, 5683, 5684, 5693, 5715, 5723, 5729, 5730, 5733, 5747, 5758, 5775, 5776, 5789, 5790, 5798, 5809, 5811, 5818, 5820, 5821, 5828, 5832, 5833, 5844, 5845, 5855, 5857, 5861, 5862, 5867, 5874, 5875, 5880, 5885, 5888, 5890, 5895, 5904, 5906, 5909, 5911, 5915, 5927, 5928, 5931, 5936, 5941, 5942, 5943, 5950, 5956, 5962, 5970, 5972, 5981, 5982, 5983, 5992, 6001, 6008, 6017, 6021, 6025, 6031, 6034, 6037, 6040, 6050, 6052, 6059, 6065, 6067, 6069, 6072, 6080, 6084, 6100, 6101, 6113, 6114, 6129, 6131, 6138, 6143, 6146, 6149, 6151, 6154, 6157, 6160, 6162, 6168, 6169, 6171, 6179, 6182, 6189, 6196, 6200, 6204, 6211, 6212, 6216, 6217, 6223, 6241, 6250, 6259, 6260, 6265, 6267, 6269, 6283, 6284, 6287, 6288, 6290, 6302, 6308, 6315, 6323, 6340, 6347, 6352, 6359, 6371, 6374, 6375, 6377, 6378, 6390, 6398, 6404, 6407, 6413, 6416, 6418, 6422, 6426, 6427, 6438, 6445, 6461, 6464, 6476, 6477, 6487, 6488, 6489, 6490, 6495, 6503, 6509, 6522, 6528, 6531, 6537, 6544, 6547, 6548, 6549, 6551, 6556, 6567, 6570, 6571, 6576, 6581, 6582, 6584, 6586, 6588, 6590, 6594, 6603, 6607, 6611, 6613, 6618, 6631, 6633, 6635, 6637, 6643, 6645, 6647, 6652, 6663, 6664, 6667, 6671, 6672, 6673, 6674, 6675, 6679, 6684, 6690, 6693, 6702, 6703, 6707]
VAL672_NUMS = [37, 57, 65, 84, 91, 108, 113, 119, 124, 130, 132, 134, 145, 169, 174, 177, 193, 213, 221, 224, 225, 247, 251, 267, 273, 281, 300, 307, 309, 316, 317, 322, 356, 360, 363, 370, 388, 390, 401, 428, 432, 442, 481, 500, 508, 517, 528, 537, 551, 558, 567, 584, 594, 601, 603, 613, 641, 649, 650, 670, 673, 675, 698, 703, 756, 775, 778, 785, 787, 792, 797, 803, 807, 827, 833, 842, 846, 858, 863, 870, 871, 873, 893, 909, 912, 913, 937, 944, 968, 983, 985, 1000, 1005, 1006, 1012, 1034, 1039, 1042, 1045, 1049, 1061, 1072, 1079, 1080, 1092, 1094, 1097, 1106, 1117, 1118, 1120, 1127, 1139, 1145, 1149, 1152, 1155, 1156, 1166, 1171, 1172, 1193, 1199, 1200, 1201, 1213, 1272, 1275, 1297, 1298, 1324, 1325, 1328, 1329, 1333, 1341, 1342, 1348, 1361, 1372, 1392, 1402, 1420, 1433, 1437, 1458, 1461, 1471, 1483, 1512, 1525, 1528, 1530, 1533, 1537, 1543, 1560, 1578, 1593, 1614, 1641, 1650, 1659, 1663, 1705, 1725, 1730, 1734, 1744, 1753, 1755, 1758, 1762, 1770, 1803, 1825, 1827, 1828, 1837, 1861, 1867, 1888, 1891, 1892, 1893, 1894, 1903, 1919, 1930, 1931, 1954, 1960, 1977, 1981, 1982, 1992, 1997, 2014, 2018, 2031, 2046, 2049, 2060, 2079, 2085, 2087, 2105, 2120, 2133, 2144, 2149, 2172, 2173, 2177, 2189, 2198, 2206, 2216, 2222, 2241, 2263, 2288, 2308, 2313, 2315, 2324, 2344, 2346, 2349, 2352, 2353, 2356, 2363, 2364, 2366, 2368, 2384, 2415, 2436, 2449, 2458, 2467, 2470, 2481, 2487, 2498, 2514, 2517, 2527, 2538, 2546, 2548, 2561, 2565, 2583, 2586, 2590, 2628, 2662, 2668, 2669, 2697, 2704, 2725, 2730, 2731, 2736, 2741, 2743, 2749, 2771, 2777, 2784, 2797, 2812, 2816, 2817, 2830, 2836, 2852, 2864, 2865, 2907, 2925, 2928, 2931, 2937, 2967, 2972, 2993, 3003, 3011, 3013, 3045, 3054, 3073, 3076, 3083, 3086, 3100, 3113, 3115, 3121, 3128, 3154, 3194, 3199, 3202, 3228, 3230, 3239, 3244, 3250, 3266, 3280, 3289, 3298, 3323, 3328, 3333, 3349, 3359, 3361, 3364, 3374, 3383, 3408, 3441, 3447, 3448, 3451, 3452, 3455, 3458, 3470, 3489, 3541, 3549, 3580, 3582, 3583, 3594, 3599, 3616, 3622, 3623, 3632, 3658, 3663, 3665, 3673, 3676, 3680, 3688, 3696, 3698, 3699, 3718, 3757, 3792, 3802, 3809, 3821, 3832, 3875, 3888, 3890, 3899, 3912, 3916, 3920, 3930, 3943, 3947, 3973, 3977, 3981, 3985, 3996, 4018, 4039, 4044, 4068, 4074, 4082, 4106, 4117, 4125, 4130, 4132, 4149, 4167, 4186, 4191, 4193, 4195, 4204, 4208, 4213, 4229, 4254, 4257, 4264, 4267, 4291, 4293, 4297, 4305, 4306, 4310, 4315, 4332, 4341, 4354, 4367, 4372, 4380, 4392, 4398, 4401, 4402, 4404, 4408, 4409, 4419, 4430, 4439, 4445, 4450, 4454, 4461, 4468, 4471, 4478, 4491, 4497, 4518, 4528, 4531, 4540, 4553, 4569, 4584, 4596, 4600, 4602, 4603, 4605, 4614, 4630, 4639, 4649, 4657, 4659, 4668, 4670, 4701, 4708, 4719, 4720, 4744, 4748, 4754, 4760, 4764, 4771, 4786, 4789, 4790, 4801, 4802, 4808, 4812, 4814, 4822, 4824, 4831, 4836, 4843, 4847, 4859, 4866, 4868, 4874, 4878, 4889, 4914, 4917, 4918, 4925, 4944, 4954, 4968, 4971, 5009, 5013, 5045, 5058, 5077, 5080, 5109, 5126, 5145, 5148, 5155, 5158, 5167, 5169, 5182, 5187, 5188, 5209, 5220, 5245, 5254, 5261, 5266, 5268, 5269, 5285, 5289, 5290, 5308, 5331, 5340, 5361, 5371, 5374, 5384, 5420, 5422, 5452, 5459, 5461, 5478, 5486, 5488, 5495, 5499, 5506, 5509, 5518, 5519, 5521, 5524, 5525, 5526, 5532, 5534, 5536, 5545, 5547, 5555, 5556, 5560, 5564, 5610, 5618, 5622, 5629, 5653, 5683, 5684, 5723, 5729, 5730, 5747, 5789, 5790, 5811, 5818, 5832, 5833, 5844, 5845, 5855, 5857, 5861, 5867, 5880, 5885, 5888, 5895, 5904, 5906, 5928, 5936, 5950, 5962, 5972, 5982, 5992, 6017, 6025, 6031, 6037, 6040, 6050, 6069, 6100, 6101, 6113, 6131, 6143, 6149, 6154, 6157, 6160, 6162, 6169, 6171, 6179, 6189, 6196, 6212, 6216, 6217, 6241, 6259, 6260, 6267, 6283, 6284, 6287, 6288, 6290, 6302, 6308, 6315, 6323, 6340, 6347, 6359, 6371, 6375, 6378, 6390, 6404, 6413, 6416, 6418, 6422, 6426, 6427, 6438, 6445, 6464, 6476, 6522, 6528, 6547, 6548, 6549, 6551, 6556, 6576, 6584, 6590, 6603, 6611, 6618, 6637, 6645, 6647, 6663, 6671, 6673, 6675, 6684, 6693, 6702, 6703]
manifest = pd.DataFrame(
    [{"id": f"train_{n:04d}.jpg", "source": "train"} for n in TRAIN_NUMS]
    + [{"id": f"dev_{n:04d}.jpg", "source": "dev"} for n in DEV_NUMS])
HOLDOUT = {f"train_{n:04d}.jpg" for n in HOLDOUT_NUMS}
VAL672 = {f"train_{n:04d}.jpg" for n in VAL672_NUMS}
assert len(HOLDOUT) == 1300 and VAL672 <= HOLDOUT

train_df = pd.read_csv(DATA_ROOT / "train.csv", keep_default_na=False)
dev_df = pd.read_csv(DATA_ROOT / "dev.csv", keep_default_na=False)
fit_rows = qlora.rows_from_manifest(manifest, train_df, dev_df, HOLDOUT)
random.Random(42).shuffle(fit_rows)
order = []
for epoch in range(EPOCHS):
    rows = fit_rows[:MAX_TRAIN_ROWS] if MAX_TRAIN_ROWS else fit_rows
    order += random.Random(42 + epoch).sample(rows, len(rows))
print(f"학습 {len(fit_rows)}문항 / 정답 분포 {pd.Series([r['answer'] for r in fit_rows]).value_counts().sort_index().to_dict()}")
print(f"옵티마이저 스텝 {len(order) // GRAD_ACCUM} (유효 배치 {GRAD_ACCUM}, epoch {EPOCHS})")

## 5. 원본 모델 다운로드

In [ ]:
from huggingface_hub import snapshot_download

if not (BASE_DIR / "config.json").is_file():
    snapshot_download(repo_id="Qwen/Qwen3.5-27B", local_dir=str(BASE_DIR))
vr.check_weights(BASE_DIR)
print("원본 준비:", BASE_DIR)

## 6. 학습

첫 스텝과 10스텝마다 손실과 남은 시간을 출력합니다. 손실이 처음부터 작은 게 정상입니다
(제로샷이 이미 97% 가까이 맞힘). 학습 신호는 모델과 사람이 어긋나는 문항에서 오므로 `max`(16문항 중
가장 큰 손실)가 줄어드는지를 보세요.

**끊기면:** 2·3·4·5번 셀을 다시 실행한 뒤 이 셀을 실행하면 Drive 체크포인트에서 이어집니다.
이미 학습이 끝났으면(`adapter/`가 있으면) 건너뜁니다.

In [ ]:
if (ADAPTER_DIR / "adapter_config.json").is_file():
    print("학습이 이미 끝났습니다:", ADAPTER_DIR)
else:
    model, proc = vr.load_vlm(BASE_DIR, four_bit=(QUANT == "nf4"), dtype=DTYPE)
    model = qlora.attach_lora(model, r=LORA_R, alpha=LORA_ALPHA, dropout=LORA_DROPOUT)
    model.print_trainable_parameters()
    torch.cuda.reset_peak_memory_stats()
    try:
        qlora.train_loop(model, proc, order, grad_accum=GRAD_ACCUM, lr=LR, warmup_ratio=WARMUP_RATIO,
                         max_grad_norm=MAX_GRAD_NORM, save_every=SAVE_EVERY, ckpt_dir=CKPT_DIR,
                         log_path=TRAIN_LOG, consensus=CONSENSUS)
        model.save_pretrained(str(ADAPTER_DIR))
        (DRIVE_RUN / "train_config.json").write_text(json.dumps(dict(
            config=CONFIG, quant=QUANT, consensus=CONSENSUS, lora_r=LORA_R, lora_alpha=LORA_ALPHA,
            lora_dropout=LORA_DROPOUT, lr=LR, grad_accum=GRAD_ACCUM, epochs=EPOCHS,
            train_rows=len(fit_rows), max_train_rows=MAX_TRAIN_ROWS,
            peak_gib=torch.cuda.max_memory_allocated() / 1024**3), indent=2), encoding="utf-8")
        print(f"학습 완료 / GPU peak {torch.cuda.max_memory_allocated() / 1024**3:.1f} GiB / 어댑터: {ADAPTER_DIR}")
    finally:
        del model, proc
        gc.collect(); torch.cuda.empty_cache()

## 7. 병합

어댑터를 bf16 원본에 합쳐 로컬 디스크(`/content/merged/…`, 약 54 GiB)에 저장합니다. vLLM이 이 폴더를
그대로 올립니다. Drive에는 올리지 않습니다(어댑터만 있으면 언제든 다시 합칠 수 있음).

In [ ]:
if (MERGED_DIR / "config.json").is_file():
    print("이미 병합됨:", MERGED_DIR)
else:
    t0 = time.time()
    qlora.merge_and_save(BASE_DIR, ADAPTER_DIR, MERGED_DIR, dtype=DTYPE)
    gc.collect(); torch.cuda.empty_cache()
    print(f"병합 {time.time() - t0:.0f}초 ->", MERGED_DIR)

## 8. 평가 — holdout 1,300

**vLLM 적재에서 메모리 오류가 나면** *런타임 → 세션 다시 시작* 후 2·3·4번 셀과 이 셀만 실행하세요.
병합한 모델은 로컬 디스크에 남아 있습니다.

비교 기준은 E6 1단계(27B 제로샷)의 같은 holdout 예측입니다(Drive `output/exp_E6_zeroshot_train`).
합의 지시문으로 학습했다면 E6 3단계(`exp_E6_consensus_train`)와 비교합니다.

In [ ]:
from transformers import AutoProcessor
from vllm import LLM, SamplingParams

llm = LLM(model=str(MERGED_DIR), dtype="bfloat16", max_model_len=4096, gpu_memory_utilization=0.90,
          limit_mm_per_prompt={"image": 1}, seed=42)
proc = AutoProcessor.from_pretrained(str(MERGED_DIR))
SAMPLING = SamplingParams(temperature=0.0, max_tokens=MAX_NEW_TOKENS, logprobs=20, seed=42)

gold = train_df[train_df.id.isin(HOLDOUT)]
tuned = ve.run(llm, proc, SAMPLING, gold.drop(columns=["answer"]).to_dict("records"),
               Path("/content/out") / RUN_NAME / "holdout", DRIVE_RUN / "holdout",
               consensus=CONSENSUS, desc="holdout 1,300")
t = gold[["id", "answer"]].merge(tuned[["id", "pred", "conf"]], on="id", validate="one_to_one")
print(f"\nE7-{CONFIG} holdout {(t.pred == t.answer).mean():.4f} ({int((t.pred == t.answer).sum())}/1300)"
      f" / val 672 {(t[t.id.isin(VAL672)].pred == t[t.id.isin(VAL672)].answer).mean():.4f}"
      f" / INVALID {int((t.pred == 'INVALID').sum())}")

base_name = "exp_E6_consensus_train" if CONSENSUS else "exp_E6_zeroshot_train"
base_path = DRIVE_BASE / "output" / base_name / "predictions.jsonl"
if base_path.is_file():
    base = ve.read(base_path)
    ve.paired(gold, base[base.id.isin(HOLDOUT)], tuned, "E6 제로샷", f"E7-{CONFIG}")
    ve.paired(gold[gold.id.isin(VAL672)], base, tuned, "E6 제로샷(val 672)", f"E7-{CONFIG}(val 672)")
else:
    print("E6 결과가 없어 비교를 건너뜁니다:", base_path)

## 9. 제출 (선택)

위 비교에서 E7이 E6 제로샷보다 **holdout에서 유의하게 좋을 때만** `SUBMIT = True`로 두고 실행하세요.
test 6,714문항을 같은 모델로 풀어 Drive `output/<실행 이름>/test/`에 제출 파일을 만듭니다(약 22분).

In [ ]:
SUBMIT = False

if not SUBMIT:
    print("SUBMIT = False: 제출을 건너뜁니다")
else:
    test_df = pd.read_csv(DATA_ROOT / "test.csv", keep_default_na=False)
    sample_df = pd.read_csv(DATA_ROOT / "sample_submission.csv", keep_default_na=False)
    assert list(test_df.id) == list(sample_df.id)
    res = ve.run(llm, proc, SAMPLING, test_df.to_dict("records"),
                 Path("/content/out") / RUN_NAME / "test", DRIVE_RUN / "test", consensus=CONSENSUS,
                 desc="test 6,714")
    res, n_fill = ve.fill_invalid(res)
    sub = test_df[["id"]].merge(res[["id", "pred"]], on="id", how="left", validate="one_to_one")
    assert sub.pred.isin(ce.CHOICES).all() and list(sub.id) == list(sample_df.id)
    share = sub.pred.value_counts(normalize=True).reindex(ce.CHOICES).fillna(0)
    print(f"폴백 {n_fill}건 / 예측 분포", share.round(3).to_dict())
    name = f"submission_qwen35-27b_lora-{CONFIG}{'-cons' if CONSENSUS else ''}-bf16-vllm.csv"
    sub.rename(columns={"pred": "answer"}).to_csv(DRIVE_RUN / "test" / name, index=False, encoding="utf-8")
    print("제출 파일:", DRIVE_RUN / "test" / name)

## 끝나면

Drive `output/<실행 이름>/`(train_log.jsonl, train_config.json, holdout/, adapter/)를 로컬 `output/`으로 받아 주세요.
`adapter/`는 수백 MB라 필요할 때만 받으면 됩니다. 구성끼리의 비교(A vs C 등)는 두 실행의 holdout 예측으로 합니다.